# Generalist Gaming Agent: A Visual Guide
### The clock, the controls and the four practice games, in pictures

This notebook explains the **Generalist Gaming Agent** competition without running a single model (sections 1 to 12): how fast the game
clock is, how big the action space is, what the four practice games ask of an agent, what the two official starter
notebooks do, and where the open space might be. Every number is computed in this notebook from the competition files
(CPU only, about a minute). **New in version 3:** the board now has public agents that score well above the host starter. Section 13 compares them and explains how to read the top score (0.291, which comes with hand-written controllers for the four practice games), section 14 reports what two agents actually **do** on the practice games (measured on the competition GPU; the games' own scores cannot be read, so no scores), and section 15 carries the top public agent as the one this notebook submits. Charts first, code collapsed: click **Show hidden code** under any figure to see how it was drawn.

> **What this is not.** Nothing is trained or tuned here. Sections 1 to 12 make **no score claims**; section 14 records behaviour in short runs on the practice games, not scores. Section 11 lists ideas;
> **none of them has been tested**, and none is claimed to help.

In [ ]:
import os, sys, math, json, itertools, textwrap, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, Circle, FancyArrowPatch
warnings.filterwarnings('ignore')

BG, INK, MUTED, RED, RULE = '#fbfaf7', '#1f2430', '#4b5563', '#c0392b', '#e5e1d8'
GAME_COL = {'asteroid_capitalist': '#4c6ef5', 'blarglike': '#2a9d8f', 'cavebots': '#c96442', 'off_line': '#e0a100'}
GAME_NAME = {'asteroid_capitalist': 'Asteroid Capitalist', 'blarglike': 'Blarglike', 'cavebots': 'Cavebots', 'off_line': 'Off Line'}
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.facecolor': BG,
                     'axes.facecolor': BG, 'savefig.facecolor': BG, 'text.color': INK, 'axes.labelcolor': INK,
                     'xtick.color': INK, 'ytick.color': INK, 'axes.edgecolor': '#9aa0a6'})

INPUT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path(os.environ['LOCAL_INPUT'])
GAMES_JSON = next(INPUT.rglob('games.json'))
COMP = GAMES_JSON.parent
PROFILE_CSV = next(INPUT.rglob('sample_games_profile.csv'))
sys.path.insert(0, str(COMP))
from gga import protocol as P          # the competition's own protocol module (pure Python)
from gga import lockstep as LS

games = json.loads(GAMES_JSON.read_text())
spec = games['games']
profile = pd.read_csv(PROFILE_CSV)
FPS, GRID_FRAMES = LS.FPS, LS.CADENCE
GRID_S = GRID_FRAMES / FPS
EPISODE_S = games['default_minutes'] * 60
N_GAMES, N_SAMPLE, N_PUBLIC_LB = 32, len(spec), 16
POINTS_PER_GAME = EPISODE_S * FPS // GRID_FRAMES
print(f'{len(spec)} sample games found: {list(spec)}')
print(f'{FPS} fps, a decision point every {GRID_FRAMES} frames ({GRID_S*1000:.0f} ms), {EPISODE_S} s per game -> {POINTS_PER_GAME} decision points')

In [ ]:
def cover():
    fig = plt.figure(figsize=(12, 3.9)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 12); ax.set_ylim(0, 3.9); ax.axis('off')
    ax.add_patch(Rectangle((0, 0), 12, 3.9, color=BG, zorder=0))
    # 32 dots: 4 known games in colour, 28 hidden
    xs0, ys0, dx = 6.7, 3.3, 0.58
    known = list(spec)
    for i in range(N_GAMES):
        r, c = divmod(i, 8)
        x, y = xs0 + c * dx, ys0 - r * dx
        if i < len(known):
            ax.add_patch(Circle((x, y), 0.23, color=GAME_COL[known[i]], zorder=3))
            ax.add_patch(Circle((x, y), 0.31, fill=False, ec=GAME_COL[known[i]], lw=1.5, alpha=.45, zorder=2))
        else:
            ax.add_patch(Circle((x, y), 0.23, color='#d9d5ca', zorder=3))
            ax.text(x, y - .02, '?', ha='center', va='center', fontsize=12, color='#9a958a', fontweight='bold', zorder=4)
    ax.text(xs0 - .1, 0.98, '4 games you can play at home,  28 you meet only at evaluation', fontsize=10.5, color=MUTED)
    # frame ticks: 20 per second, a decision point every 4th
    for k in range(21):
        h = 0.34 if k % GRID_FRAMES == 0 else 0.17
        ax.plot([xs0 - .1 + k * 0.255] * 2, [0.3, 0.3 + h], color=RED if k % GRID_FRAMES == 0 else '#b9b4a8', lw=2.2 if k % GRID_FRAMES == 0 else 1.2, solid_capstyle='round')
    ax.text(xs0 - .1, 0.08, f'one second of game time: {FPS} frames, a decision point every {GRID_FRAMES} (every {GRID_S*1000:.0f} ms)', fontsize=9.5, color=MUTED)
    ax.text(0.5, 2.85, 'Playing games\nyou have never seen', fontsize=36, fontweight='bold', va='center', linespacing=1.04)
    ax.text(0.53, 1.62, 'A visual guide to the Generalist Gaming Agent competition', fontsize=15, color=MUTED, va='center')
    ax.plot([0.55, 1.65], [1.18, 1.18], color=RED, lw=3.5, solid_capstyle='round')
    ax.text(0.53, 0.62, 'The game keeps running while your agent thinks.\nWhat does that do to the design of an agent?', fontsize=11.5, color=MUTED, va='center', linespacing=1.5)
    plt.show()
cover()

In [ ]:
from IPython.display import HTML, display
def box(title, html, color='#2a9d8f'):
    display(HTML(f'<div style="border-left:6px solid {color};background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:{color}">{title}</b><br>{html}</div>'))

def delay_grid_points(latency_s):
    # a reply takes effect at the first decision point after it arrives; one that arrives within 3 frames
    # of its observation lands at the very next point (protocol.rules()): grid index = floor(latency / 0.2 s) + 1
    return np.floor(np.asarray(latency_s) / GRID_S + 1e-9).astype(int) + 1

STARTER_REPLY_S = 1.25                                  # REPLY_SECONDS in both official starters
lag_pts = int(delay_grid_points(STARTER_REPLY_S))
starter_decisions = POINTS_PER_GAME / lag_pts
total_game_min = N_GAMES * EPISODE_S / 60
box('Key numbers, computed in this notebook',
    '<ul style="margin:6px 0 0 18px">'
    f'<li>A game is <b>{POINTS_PER_GAME:,}</b> decision points ({EPISODE_S//60} min x {FPS} fps / {GRID_FRAMES}). An agent that needs <b>{STARTER_REPLY_S} s</b> per reply (what both official starters pad to) acts at about <b>{starter_decisions:.0f}</b> of them, <b>{starter_decisions/POINTS_PER_GAME:.0%}</b>, and every reply arrives <b>{lag_pts*GRID_S:.1f} s</b> after the frame it was based on.</li>'
    f'<li>All {N_GAMES} games add up to <b>{total_game_min/60:.1f} hours</b> of game time per submission.</li>'
    f'<li>Each game is worth <b>1/{N_PUBLIC_LB} = {1/N_PUBLIC_LB:.4f}</b> of the public leaderboard; the 0.093 shown by the Gemma starter is about <b>{0.093*N_PUBLIC_LB:.1f} games</b> worth of score.</li>'
    f'<li>The four practice games need <b>four different control schemes</b>; no single set of keys works for all of them.</li>'
    '</ul><span style="color:#4b5563">Numbers come from the competition files and the harness code, not from leaderboard experiments.</span>', '#c96442')

**Contents:** 1 the competition at a glance, 2 where it sits among other competitions, 3 the game clock, 4 the action space, 5 the four practice games,
6 what a score means, 7 inside the game files, 8 the two official starters, 9 compute, time and complexity, 10 who should join (and who should not),
11 ideas nobody has tested here, 12 limits and credits, 13 the host's starter, unchanged.

## 1. The competition at a glance
One offline agent plays **32 real-time browser games across 10 genres**, each for 10 minutes of game time. It sees raw screen
frames and answers with mouse and keyboard input. Four games ship with the data; the other 28 stay hidden until evaluation
and arrive with only a short brief of the goal and controls. The public leaderboard averages 16 of the 32 games (including the
four practice games); the private leaderboard averages the other 16 and decides the ranking.

In [ ]:
def at_a_glance():
    fig = plt.figure(figsize=(12, 6.6)); gs = fig.add_gridspec(2, 1, height_ratios=[1.05, 1], hspace=.25)
    ax = fig.add_subplot(gs[0]); ax.set_xlim(0, 12); ax.set_ylim(0, 3); ax.axis('off')
    tiles = [('32', 'games, back to back'), ('10', 'genres'), ('4 + 28', 'practice + hidden games'), ('10 min', 'of game time per game'),
             ('16 + 16', 'public + private board'), ('5', 'submissions per day'), ('9 h', 'notebook run-time limit'), ('$50,000', 'prizes, top 5 places')]
    cols = ['#4c6ef5', '#2a9d8f', '#c96442', '#e0a100']
    for i, (big, small) in enumerate(tiles):
        r, c = divmod(i, 4); x, y = 0.12 + c * 2.97, 1.62 - r * 1.5
        ax.add_patch(FancyBboxPatch((x, y), 2.75, 1.25, boxstyle='round,pad=0.02,rounding_size=0.08', fc='white', ec=RULE, lw=1.2))
        ax.add_patch(Rectangle((x + .02, y + .05), 0.09, 1.15, color=cols[c], lw=0))
        ax.text(x + .3, y + .72, big, fontsize=26, fontweight='bold', va='center', color=INK)
        ax.text(x + .3, y + .28, small, fontsize=10.5, va='center', color=MUTED)
    # timeline
    ax2 = fig.add_subplot(gs[1]); ax2.set_xlim(-2, 104); ax2.set_ylim(-1.2, 3.7); ax2.axis('off')
    d_merge, d_final = 91, 98                                   # days after the 23 Nov 2026 start (22 Feb and 1 Mar 2027)
    ax2.add_patch(FancyBboxPatch((0, 1.55), d_final, 0.42, boxstyle='round,pad=0,rounding_size=0.2', fc='#e9e5da', ec='none'))
    ax2.add_patch(FancyBboxPatch((0, 1.55), d_merge, 0.42, boxstyle='round,pad=0,rounding_size=0.2', fc='#c96442', ec='none', alpha=.9))
    for d, lab, up in [(0, 'Start\n23 Nov 2026', True), (d_merge, 'Entry + team-merger deadline\n22 Feb 2027', True), (d_final, 'Final submission\n1 Mar 2027', False)]:
        ax2.plot([d, d], [1.45, 2.07], color=INK, lw=1.6)
        ax2.text(d - (0 if d == 0 else 1.2), 2.27 if up else 0.95, lab, ha='left' if d == 0 else 'right' if d == d_merge else 'left', va='center' if not up else 'bottom', fontsize=10.5, color=INK) if d != d_final else ax2.text(d + 1.2, 1.76, lab, ha='left', va='center', fontsize=10.5)
    ax2.text(d_merge / 2, 1.76, f'{d_final} days to build, tune, and submit', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
    ax2.text(0, 3.5, 'Timeline', fontsize=13, fontweight='bold', va='center')
    prizes = [16000, 12000, 9000, 7000, 6000]
    for i, p in enumerate(prizes):
        y = 0.35 - i * 0.30
        ax2.add_patch(Rectangle((11, y), p / 16000 * 40, 0.2, color='#4c6ef5', alpha=.85 - i * .1, lw=0))
        ax2.text(10, y + .1, f'{i + 1}', ha='right', va='center', fontsize=9.5, color=MUTED)
        ax2.text(11 + p / 16000 * 40 + .8, y + .1, f'${p:,}', va='center', fontsize=9.5)
    ax2.text(0, 0.6, 'Prizes', fontsize=13, fontweight='bold', va='center')
    ax2.text(62, 0.35, 'Winning code is licensed MIT; winners also owe\na write-up that lets others reproduce the result.', fontsize=10, color=MUTED, va='center', linespacing=1.4)
    plt.show()
at_a_glance()

## 2. Where it sits among other competitions
Five properties make this competition unusual: the agent sees **only pixels**, the game **keeps running while it thinks**, most test games are **unseen**, you **submit code** that the host runs offline,
and a **vision-language model can drive it**. The chart marks which of these a handful of well-known agent competitions share. It is our reading of their public descriptions
(full circle = yes, half = partly, empty = no); check each competition's own page before relying on a cell.

In [ ]:
from matplotlib.patches import Wedge
def similar_figure():
    cols = ['Pixels\nonly', 'Clock keeps\nrunning', 'Unseen games\nor levels', 'Submit code,\nhost runs it', 'Vision-language\nfriendly']
    rows = [('Generalist Gaming Agent', 'real-time browser games', [2, 2, 2, 2, 2], '#c96442'),
            ('ARC Prize 2026: ARC-AGI-3', 'turn-based puzzle games on 64 x 64 colour grids', [1, 0, 2, 2, 2], '#4c6ef5'),
            ('Kaggle simulation contests', 'Lux AI, Halite, Hungry Geese, Football', [0, 0, 1, 2, 1], '#2a9d8f'),
            ('Minecraft agents (MineRL, BASALT)', 'pixels, long tasks, stepped simulator', [2, 0, 1, 2, 1], '#8e6bbf'),
            ('Level generalisation contests', 'Procgen, Obstacle Tower, Gym Retro', [2, 0, 2, 2, 0], '#e0a100'),
            ('NetHack Challenge (NeurIPS 2021)', 'roguelike, symbolic screen, turn-based', [0, 0, 1, 2, 1], '#6b6b6b')]
    fig, ax = plt.subplots(figsize=(12, 5.8)); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(-1.0, len(rows) + 1.2)
    x0, dx = 5.0, 1.5
    for j, c in enumerate(cols): ax.text(x0 + j * dx + .2, len(rows) + .45, c, ha='center', va='bottom', fontsize=9.4, fontweight='bold', linespacing=1.25)
    for i, (n, sub, v, col) in enumerate(rows):
        y = len(rows) - 1 - i
        if i % 2 == 0: ax.add_patch(Rectangle((0, y - .45), 12, .9, color='#f3f0e8', lw=0, zorder=0))
        ax.add_patch(Rectangle((0.05, y - .38), .08, .76, color=col, lw=0))
        ax.text(.3, y + .13, n, va='center', fontsize=11, fontweight='bold'); ax.text(.3, y - .22, sub, va='center', fontsize=8.8, color=MUTED)
        for j, s in enumerate(v):
            cx = x0 + j * dx + .2
            ax.add_patch(Circle((cx, y), .24, fc='white', ec=col, lw=1.6))
            if s == 2: ax.add_patch(Circle((cx, y), .24, fc=col, ec=col, lw=1.6))
            if s == 1: ax.add_patch(Wedge((cx, y), .24, 90, 270, fc=col, ec=col, lw=1.6))
    ax.text(0, -.85, 'Our reading of public descriptions, not an official comparison. ARC-AGI-3: the board is a small coloured grid rather than a photo-like frame.', fontsize=9, color=MUTED)
    plt.show()
similar_figure()

<div style="border-left:6px solid #4c6ef5;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#4c6ef5">The closest relative, and what is new</b><br><b>ARC-AGI-3</b> is the nearest cousin: unseen interactive games, no instructions, a score tied to a human baseline, run as a Kaggle code competition. Skills carry over: working out an unknown rule from a few interactions, and remembering it within a game. The big difference is the clock: <b>ARC-AGI-3 waits for you, this one does not.</b> The older level-generalisation and Minecraft contests had pixels and held-out levels but were stepped simulators built before vision-language models. <b>Unseen games, a running clock, raw pixels and a model-friendly runtime together</b> are what no earlier contest on this list combined.</div>

## 3. The game clock: the world does not wait
The game runs at **20 frames per second** with a **decision point every 4 frames (200 ms)**. A reply takes effect at the first
decision point after it arrives, and the next observation is that point. So a slow reply is not dropped, it is just **late**.
The left chart shows how a reply of 450 ms lands; the right charts turn the harness rule into two numbers per reply latency:
how stale the action is, and how many decisions fit into a game.

In [ ]:
def clock_figure():
    fig = plt.figure(figsize=(12, 6.6)); gs = fig.add_gridspec(2, 2, height_ratios=[0.6, 1.3], hspace=.32, wspace=.22)
    ax = fig.add_subplot(gs[0, :]); ax.set_xlim(-0.5, 25.5); ax.set_ylim(-1.3, 2.9); ax.axis('off')
    for f in range(25):
        point = f % GRID_FRAMES == 0
        ax.add_patch(Rectangle((f, 0.3), 0.86, 0.9, fc='#f3d9cf' if point else 'white', ec='#c96442' if point else RULE, lw=1.6 if point else 1))
        ax.text(f + .43, 0.75, str(f), ha='center', va='center', fontsize=8.5, color=INK if point else MUTED)
    ax.text(-.5, 1.7, 'Frames of one game (20 per second). Decision points are the highlighted frames.', fontsize=12, fontweight='bold', va='center')
    # example: observation at frame 0, reply arrives at frame 9 (450 ms), takes effect at 12
    ax.add_patch(FancyArrowPatch((0.43, 0.2), (9.43, 0.2), arrowstyle='-|>', mutation_scale=14, color='#4c6ef5', lw=2, connectionstyle='arc3,rad=-0.25'))
    ax.text(3.3, -.78, 'thinks for 450 ms (9 frames)', ha='center', fontsize=10.5, color='#4c6ef5', fontweight='bold')
    ax.add_patch(FancyArrowPatch((9.6, 0.2), (12.4, 0.2), arrowstyle='-|>', mutation_scale=14, color=RED, lw=2, connectionstyle='arc3,rad=-0.5'))
    ax.text(16.5, -.78, 'action lands at frame 12: 600 ms after its picture', ha='center', fontsize=10.5, color=RED, fontweight='bold')
    ax.text(0.43, 2.45, 'sees frame 0', ha='center', fontsize=10, color=INK)
    ax.text(12.43, 2.45, 'acts + sees frame 12', ha='center', fontsize=10, color=INK)
    ax.plot([.43, .43], [1.28, 2.3], color=INK, lw=1); ax.plot([12.43, 12.43], [1.28, 2.3], color=INK, lw=1)

    lat = np.linspace(0.02, 6, 600)
    lag = delay_grid_points(lat) * GRID_S
    dec = POINTS_PER_GAME / delay_grid_points(lat)
    ax2 = fig.add_subplot(gs[1, 0]); ax2.plot(lat, lag, color='#c96442', lw=2.6)
    ax2.set_xlabel('time the agent takes per reply (s)'); ax2.set_ylabel('age of the picture when the action lands (s)')
    ax2.set_title('Every reply lands late, in 200 ms steps', loc='left', fontsize=13, fontweight='bold')
    ax2.plot([STARTER_REPLY_S], [lag_pts * GRID_S], 'o', color=INK, ms=9); ax2.annotate(f'both official starters pad\nto {STARTER_REPLY_S} s: lands {lag_pts*GRID_S:.1f} s late', (STARTER_REPLY_S, lag_pts * GRID_S), (2.2, 0.55), fontsize=10, arrowprops=dict(arrowstyle='-', color=MUTED))
    ax2.grid(alpha=.35, color=RULE)
    ax3 = fig.add_subplot(gs[1, 1]); ax3.plot(lat, dec, color='#4c6ef5', lw=2.6); ax3.set_yscale('log')
    ax3.set_xlabel('time the agent takes per reply (s)'); ax3.set_ylabel('decisions per 10-minute game')
    ax3.set_title('...and the number of decisions collapses', loc='left', fontsize=13, fontweight='bold')
    ax3.plot([STARTER_REPLY_S], [starter_decisions], 'o', color=INK, ms=9)
    ax3.annotate(f'{starter_decisions:.0f} of {POINTS_PER_GAME:,} points', (STARTER_REPLY_S, starter_decisions), (2.3, 900), fontsize=10, arrowprops=dict(arrowstyle='-', color=MUTED))
    ax3.axhline(POINTS_PER_GAME, color=MUTED, lw=1, ls='--'); ax3.text(5.95, POINTS_PER_GAME * 1.1, 'a reply at every point: 3,000', ha='right', fontsize=9.5, color=MUTED)
    ax3.set_ylim(40, 4500); ax3.grid(alpha=.35, color=RULE, which='both')
    plt.show()
clock_figure()

<div style="border-left:6px solid #6b7280;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#6b7280">In plain words</b><br>The slower the model, the older the world it reacts to. A model that takes 1.25 s looks at a picture, thinks, and acts on a world that has moved on for 1.4 s. That is fine for a menu or a turn-based dungeon and fatal for a ship that dodges asteroids. The harness also gives every agent a way to act <i>while it thinks</i>: the <code>sequence</code> tool runs a timed program of up to 200 steps (10 s) on its own, and <code>skip</code> changes nothing.</div>

### The whole run: how much game time is one submission?
Every submission plays all 32 games back to back. Startup (installs, loading a model, `Agent.__init__`) must fit in 15 minutes and
one decision may take at most 90 seconds, or that game ends with its best score so far.

In [ ]:
def budget_figure():
    startup, limit = 15, 9 * 60
    games_min = N_GAMES * EPISODE_S / 60
    slack = limit - startup - games_min
    fig, ax = plt.subplots(figsize=(12, 2.6)); ax.axis('off'); ax.set_xlim(0, limit); ax.set_ylim(-1.0, 2.6)
    parts = [(startup, 'startup\n<= 15 min', '#e0a100'), (games_min, f'{N_GAMES} games x {EPISODE_S//60} min = {games_min/60:.1f} h of game time', '#c96442'), (slack, f'slack inside the 9 h limit:\nup to {slack//60:.0f} h {slack%60:.0f} min', '#d9d5ca')]
    x = 0
    for w, lab, col in parts:
        ax.add_patch(Rectangle((x, 0.3), w - 2, 1, fc=col, ec='none'))
        if w < 60: ax.text(x + w / 2 - 1, 1.5, lab, ha='left', va='bottom', fontsize=10.5, fontweight='bold', color=INK)
        else: ax.text(x + w / 2 - 1, 0.8, lab, ha='center', va='center', fontsize=10.5, fontweight='bold', color='white' if col != '#d9d5ca' else INK)
        x += w
    for m in range(0, limit + 1, 60):
        ax.plot([m, m], [0.1, 0.3], color=MUTED, lw=1); ax.text(m, -0.12, f'{m//60} h', ha='center', fontsize=9.5, color=MUTED)
    ax.text(0, 2.3, 'One submission, drawn to scale. The leaderboard takes about 6 hours to update after you submit.', fontsize=12, fontweight='bold')
    plt.show()
budget_figure()

## 4. The action space: eleven tools, an enormous number of moves
The harness exposes one universal interface for all games: pointer plus keyboard, in eleven tools. The chart counts how many
distinct actions each tool allows for a 1600 x 1000 viewport (the default). The counts use the tool schemas in
`gga/protocol.py`; they are upper bounds on what is *expressible*, not on what is *useful*.

In [ ]:
from math import comb
W, H = games['default_viewport']
cand = list(P.NAMED_KEYS) + [chr(c) for c in range(32, 127)]
KEYS = sorted({P.normalize_key(k) for k in cand if P.valid_key(k)})
K, PX = len(KEYS), W * H
MS, WHEEL = P.MAX_HOLD_MS, 2 * 3000 + 1
subsets = sum(comb(K, j) for j in range(1, P.MAX_KEYS + 1))        # 1..8 keys held at once
log10 = math.log10
rows = [
    ('click',      'pointer', PX * 3),
    ('move',       'pointer', PX),
    ('drag',       'pointer', PX * PX * MS * 3),
    ('scroll',     'pointer', PX * WHEEL * WHEEL),
    ('press_key',  'keyboard', K * MS),
    ('hold_keys',  'keyboard', subsets * MS),
    ('key_down',   'keyboard', K),
    ('key_up',     'keyboard', K),
    ('wait',       'timing', MS * 2),
    ('skip',       'timing', 1),
]
step = (subsets + 1) * 8 * (PX + 1) * (MS + 1)                      # keys x buttons x optional pointer x duration
rows.append(('sequence', 'program', None))
action_space = pd.DataFrame([{'tool': t, 'group': g, 'log10_actions': (log10(n) if n else P.MAX_STEPS * log10(step)),
                              'actions_or_note': (f'{n:,}' if n else f'about 10^{P.MAX_STEPS * log10(step):.0f} (up to {P.MAX_STEPS} steps)')} for t, g, n in rows])
print(f'{K} distinct keys, {PX:,} pixels, {subsets:,} ways to hold 1-{P.MAX_KEYS} keys')
print(action_space.to_string(index=False))

In [ ]:
def action_figure():
    d = action_space.sort_values('log10_actions').copy(); CAP = 24.0; d['shown'] = d.log10_actions.clip(upper=CAP); col = {'pointer': '#4c6ef5', 'keyboard': '#2a9d8f', 'timing': '#9aa0a6', 'program': '#c96442'}
    fig, ax = plt.subplots(figsize=(12, 5.3))
    y = np.arange(len(d)); ax.barh(y, d.shown.clip(lower=0.05), color=[col[g] for g in d.group], height=.66)
    ax.set_yticks(y); ax.set_yticklabels(d.tool, fontsize=12, family='monospace')
    for yi, v, sh in zip(y, d.log10_actions, d.shown):
        ax.text(max(sh, .05) + .4, yi, (f'10^{v:.0f}  (bar cut off at 10^{CAP:.0f})' if v > CAP else f'10^{v:.1f}') if v > 0.3 else '1', va='center', fontsize=10.5)
    ax.axvline(log10(255), color=INK, lw=1.6, ls='--'); ax.text(log10(255) + .3, len(d) - .35, 'Drex starter: 255 choices', fontsize=10, va='bottom')
    ax.set_ylim(-.6, len(d) + .2); ax.set_xlabel('log10 of the number of distinct actions (one grid line = 10x more)'); ax.set_xlim(0, CAP * 1.35)
    ax.set_title('A tool is not a move: counting what each one can say', loc='left', fontsize=14, fontweight='bold')
    from matplotlib.lines import Line2D
    ax.legend([Line2D([], [], color=c, lw=8) for c in col.values()], ['pointer tools', 'keyboard tools', 'timing', 'program (a script of steps)'], loc='lower right', frameon=False, fontsize=10.5)
    ax.grid(axis='x', alpha=.35, color=RULE); plt.show()
action_figure()

<div style="border-left:6px solid #4c6ef5;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#4c6ef5">What this means</b><br>Nobody searches this space. Every agent shrinks it first: the Drex starter reduces it to <b>255 choices</b> (59 key options, 195 click cells and <code>wait</code>), the Gemma starter lets the model write a short JSON action. The interesting design question is which <b>small, game-independent action vocabulary</b> still plays every genre. A 15 x 13 click grid over a 1600 x 1000 screen makes each cell about 107 x 77 pixels, large next to a thin on-screen line or a small button.</div>

## 5. The four practice games: four different control schemes
Our reading of each game's brief (the table behind the chart is in the companion dataset, in our own words). A filled dot means the game
needs that kind of input. Read the rows as **skills an agent must have before it can score anything**: starting a game, steering with keys,
aiming a pointer, holding a button, using hotkeys, and restarting after a game over.

In [ ]:
def controls_figure():
    feats = [('start_by', 'click', 'Starts with a mouse click'), ('start_by', 'key', 'Starts with a key press'),
             ('keyboard_steering', 1, 'Keys steer or move'), ('pointer_in_play', 1, 'Pointer aims or clicks during play'),
             ('mouse_button_held', 1, 'Mouse button must be held'), ('right_click', 1, 'Right mouse button'),
             ('build_hotkeys', 1, 'Command hotkeys (Q W E R)'), ('turn_based', 1, 'Turn-based: no clock pressure'),
             ('real_time_pressure', 1, 'Real-time pressure'), ('restart_needs_action', 1, 'Needs an action to restart after game over')]
    gids = list(profile.game_id)
    fig, ax = plt.subplots(figsize=(12, 6.4)); DX = 2.2; ax.set_xlim(-5.7, len(gids) * DX + .3); ax.set_ylim(-2.6, len(feats) + 1.1); ax.axis('off')
    for j, g in enumerate(gids):
        ax.text(j * DX + 1.0, len(feats) + .5, GAME_NAME[g].replace(' ', '\n', 1) if len(GAME_NAME[g]) > 12 else GAME_NAME[g], ha='center', va='bottom', fontsize=11, linespacing=1.0, fontweight='bold', color=GAME_COL[g])
        row = profile[profile.game_id == g].iloc[0]
        ax.text(j * DX + 1.0, len(feats) - .15, f'{row.genre}', ha='center', fontsize=9.5, color=MUTED)
    for i, (col, val, label) in enumerate(feats):
        y = len(feats) - 1 - i
        if i % 2 == 0: ax.add_patch(Rectangle((-5.7, y - .45), len(gids) * DX + 6.0, .9, color='#f3f0e8', lw=0, zorder=0))
        ax.text(-5.5, y, label, va='center', fontsize=11)
        for j, g in enumerate(gids):
            on = profile.loc[profile.game_id == g, col].iloc[0] == val
            ax.add_patch(Circle((j * DX + 1.0, y), .24, fc=GAME_COL[g] if on else 'none', ec=GAME_COL[g] if on else '#cfcabd', lw=1.6 if on else 1))
    for j, g in enumerate(gids):
        r = profile[profile.game_id == g].iloc[0]
        ax.text(j * DX + 1.0, -1.1, f'{r.viewport_w} x {r.viewport_h}', ha='center', fontsize=10.5)
        ax.text(j * DX + 1.0, -1.6, textwrap.fill(r.score_kind, 20), ha='center', va='top', fontsize=8.8, color=MUTED, linespacing=1.3)
    ax.text(-5.5, -1.1, 'Viewport (pixels)', va='center', fontsize=11); ax.text(-5.5, -1.6, 'What the score counts', va='top', fontsize=11)
    plt.show()
controls_figure()

<div style="border-left:6px solid #2a9d8f;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#2a9d8f">Reading across the rows</b><br>Only <b>one</b> row is shared by all four games: after a game over the agent has to <i>do something</i> to start again (click, any key, or a two-click flow), and every restart costs game time. Two games start with a click and two with a key, so an agent that only ever presses keys, or only ever clicks, is stuck on the first screen of half the practice games. <b>Blarglike</b> is the only turn-based game: its keys are ignored while a step is still animating, so spamming keys fast does not help.</div>

## 6. What a score means
Each game is scored from 0 to 1: `(value - floor) / (human - floor)`, clipped. The floor is the score of doing nothing, so **an agent that presses nothing scores 0**; the human baseline scores 1. The leaderboard is the plain average over games.

In [ ]:
def score_figure():
    fig = plt.figure(figsize=(12, 6.2)); gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1], wspace=.08)
    ax = fig.add_subplot(gs[0]); ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 6)
    ax.text(0, 5.6, 'What 1.0 means in each practice game', fontsize=13, fontweight='bold')
    heads = ['Game', 'Counts', 'Human = 1.0', 'Floor']
    xs = [0, 3.9, 6.5, 8.5]
    for x, h in zip(xs, heads): ax.text(x, 4.9, h, fontsize=10.5, color=MUTED, fontweight='bold')
    ax.plot([0, 10], [4.7, 4.7], color=RULE, lw=1.2)
    unit = {'asteroid_capitalist': 'dollars', 'blarglike': 'floor x nutrients', 'cavebots': 'seconds survived', 'off_line': 'time on the line'}
    for i, (g, s) in enumerate(spec.items()):
        y = 4.1 - i * .85; sc = s['scoring']
        ax.add_patch(Circle((.12, y), .13, color=GAME_COL[g])); ax.text(.4, y, GAME_NAME[g], va='center', fontsize=11, fontweight='bold')
        ax.text(xs[1], y, unit[g], va='center', fontsize=10.5)
        ax.text(xs[2], y, f"{sc['reference']:,}", va='center', fontsize=12, fontweight='bold')
        ax.text(xs[3], y, f"{sc['floor']:g}" if sc.get('floor') is not None else '0 (idle)', va='center', fontsize=10.5, color=MUTED)
    ax.text(0, 0.3, 'Cavebots: the reference equals the length of the episode, so a full\nscore means surviving all 10 minutes (a perfect survivor reads\n599 s, which is 0.998).', fontsize=9.6, color=MUTED, va='center', linespacing=1.45)
    bx = fig.add_subplot(gs[1]); bx.set_xlim(-.05, 4.4); bx.set_ylim(-1.7, 5.4); bx.axis('off')
    bx.text(0, 5.1, 'The public leaderboard is 16 squares', fontsize=13, fontweight='bold')
    for i in range(N_PUBLIC_LB):
        r, c = divmod(i, 4); x, y = c * 1.08, 3.35 - r * 1.08
        fill = 0.093 * N_PUBLIC_LB - i                       # the 0.093 of the Gemma starter, in squares
        known = i < N_SAMPLE
        bx.add_patch(FancyBboxPatch((x, y), .95, .95, boxstyle='round,pad=0,rounding_size=.08', fc='white', ec=RULE, lw=1.4))
        if fill > 0: bx.add_patch(Rectangle((x + .03, y + .03), .89, .89 * min(1, fill), color='#c96442', alpha=.85, lw=0))
        bx.text(x + .475, y + .475, '?' if not known else '', ha='center', va='center', fontsize=17, color='#c9c4b8', fontweight='bold')
    bx.text(0, 2.2 - 3.35 + 0.05 - 0.1 - 0.0, 'Each square is one game = 1/16 = 0.0625 of the score.\nThe orange part is 0.093, the Gemma starter\'s public score: about\n1.5 squares. Which 12 hidden games fill the other\nsquares is not published.', fontsize=9.6, color=MUTED, va='top', linespacing=1.45)
    plt.show()
score_figure()

## 7. Inside the game files
The agent never sees the page source, only pixels, but the files tell us what kind of game each one is. We sorted every file of the four sample
games by type and size. The engines are inferred from file names and bundled library names, so treat them as "looks like", not as an official statement.

In [ ]:
def classify(p):
    s = p.suffix.lower()
    if s == '.wasm': return 'WebAssembly'
    if s in ('.js', '.html', '.css'): return 'code (js / html / css)'
    if s in ('.png', '.jpg', '.jpeg', '.gif', '.webp'): return 'images'
    if s in ('.ogg', '.mp3', '.wav'): return 'audio'
    if s in ('.ttf', '.otf', '.woff', '.woff2'): return 'fonts'
    return 'data / other'
GAME_DIR = {'asteroid_capitalist': 'asteroid-capitalist', 'blarglike': 'blarglike', 'cavebots': 'cavebots', 'off_line': 'off-line'}
recs = []
for g, dname in GAME_DIR.items():
    for p in (COMP / 'assets').rglob('*'):
        if p.is_file() and dname in p.parts:
            recs.append({'game': g, 'kind': classify(p), 'bytes': p.stat().st_size})
assets = pd.DataFrame(recs).groupby(['game', 'kind']).bytes.sum().unstack(fill_value=0)
looks_like = {'asteroid_capitalist': 'bundled JS game (app.js + vendors.js\nwith the Phaser library inside)', 'blarglike': 'GameMaker-style HTML5 export\n(html5game/ folder)', 'cavebots': 'Go compiled to WebAssembly\n(main.wasm + wasm_exec.js)', 'off_line': 'hand-written HTML + JS + CSS\n(a few small files)'}
def asset_figure():
    kinds = ['code (js / html / css)', 'WebAssembly', 'images', 'audio', 'fonts', 'data / other']
    kc = dict(zip(kinds, ['#4c6ef5', '#c96442', '#2a9d8f', '#e0a100', '#8e6bbf', '#9aa0a6']))
    fig, ax = plt.subplots(figsize=(12, 4.9)); fig.subplots_adjust(left=.19, right=.97); order = list(GAME_DIR)[::-1]
    left = np.zeros(len(order))
    for k in kinds:
        if k not in assets.columns: continue
        v = np.array([assets.loc[g, k] / 1e6 for g in order]); ax.barh(range(len(order)), v, left=left, color=kc[k], label=k, height=.62); left += v
    ax.set_yticks(range(len(order))); ax.set_yticklabels([GAME_NAME[g] for g in order], fontsize=12, fontweight='bold')
    for i, g in enumerate(order):
        ax.text(left[i] + .25, i + .12, f'{left[i]:.1f} MB', va='center', fontsize=10.5, fontweight='bold')
        ax.text(left[i] + .25, i - .2, looks_like[g].replace('\n', ' '), va='center', fontsize=9, color=MUTED)
    ax.set_xlabel('size of the game files (MB)'); ax.set_xlim(0, left.max() * 1.9)
    ax.set_title('Four games, four ways of being built', loc='left', fontsize=14, fontweight='bold')
    ax.legend(frameon=False, ncol=3, loc='lower right', fontsize=10); ax.grid(axis='x', alpha=.35, color=RULE); plt.show()
asset_figure()

<div style="border-left:6px solid #e0a100;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#e0a100">Why look at this at all?</b><br>Because the hidden 28 games will be just as mixed: native-looking pixel games, canvas engines, compiled code. An agent cannot read the DOM or the game state, and pixels from different engines differ in size, scaling and update behaviour. Anything that relies on one game's rendering (a fixed UI position, a sprite template) is a bet on one of the 32.</div>

## 8. The two official starters, side by side
The host published two starter notebooks you can submit as they are. We read their code; the table compares what they do, which is the best way to see what they leave open.
Both run on the competition's GPU (an NVIDIA RTX Pro 6000, 96 GB).

In [ ]:
def starter_figure():
    rows = [
        ('Model', 'Drex Max (Nace.AI decision model)', 'Gemma 4 31B, 4-bit weights'),
        ('How it answers', 'One forward pass scores a fixed menu;\nit plays the most probable option', 'Writes a short JSON action\n(max 64 new tokens, greedy)'),
        ('Action vocabulary', '255 options: wait, 59 key options,\n195 click cells (15 x 13 grid)', 'click, move, press_key and a\nsequence that holds keys + mouse'),
        ('What it looks at', 'One frame (shrunk to <= 768 px),\nthe brief, the previous action', 'One frame (shrunk to <= 768 px),\nthe brief, the previous action'),
        ('Pacing', 'Every reply sent 1.25 s after its frame', 'Every reply sent 1.25 s after its frame'),
        ('Between games', 'Keeps its last action', 'Clears its last action when\nthe game clock restarts'),
        ('Public score', '0.056 (another team)', '0.093'),
    ]
    fig, ax = plt.subplots(figsize=(12, 5.9)); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(0, 7.6)
    xs = [0.1, 2.6, 7.3]
    for x, h, c in [(xs[1], 'Starter A: Drex Max decision model', '#4c6ef5'), (xs[2], 'Starter B: Gemma 4 31B (transformers)', '#c96442')]:
        ax.add_patch(FancyBboxPatch((x - .1, 6.7), 4.6, .7, boxstyle='round,pad=0,rounding_size=.1', fc=c, ec='none'))
        ax.text(x + 2.2, 7.05, h, color='white', fontweight='bold', fontsize=10.5, ha='center', va='center')
    for i, (lab, a, b) in enumerate(rows):
        y = 6.0 - i * .93
        if i % 2 == 0: ax.add_patch(Rectangle((0, y - .45), 12, .9, color='#f3f0e8', lw=0, zorder=0))
        ax.text(xs[0], y, lab, fontweight='bold', va='center', fontsize=11); ax.text(xs[1], y, a, va='center', fontsize=10.2, linespacing=1.3); ax.text(xs[2], y, b, va='center', fontsize=10.2, linespacing=1.3)
    plt.show()
starter_figure()

<div style="border-left:6px solid #c96442;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#c96442">What both of them leave open</b><br><ul style='margin:6px 0 0 18px'><li><b>Motion:</b> each decision uses one frame, although the harness hands over up to 8 earlier frames since the last look (<code>obs.history</code>).</li><li><b>Time:</b> replies are padded to 1.25 s instead of acting as fast as the model allows, and the long-running <code>sequence</code> program is used only to hold keys.</li><li><b>Memory:</b> the only state is the previous action; nothing remembers what worked in this game, or on the last screen.</li><li><b>Feedback:</b> the starters do not read the game's own score, so they cannot tell a move that helped from one that did not.</li></ul></div>

## 9. Compute, time and complexity: what taking part really costs
Stated by the competition: notebooks run **up to 9 hours** on CPU or on an **NVIDIA RTX Pro 6000 (96 GB)**, with **internet off**; startup is limited to 15 minutes, one decision to 90 seconds,
and you get **5 submissions a day**. A CPU-only agent is allowed. Local play with Playwright needs Python 3.11 or newer and reports **no score**: a score exists only after you submit and the host reruns your notebook (about 6 hours).

In [ ]:
def cost_figure():
    fig = plt.figure(figsize=(12, 8.2)); gs = fig.add_gridspec(3, 1, height_ratios=[.85, .55, 1.4], hspace=.35)
    # (a) iteration loop
    ax = fig.add_subplot(gs[0]); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(0, 3)
    steps = [('Edit the agent', 'think, change code', '#9aa0a6'), ('Test it', 'local play or notebook:\nminutes, no score', '#2a9d8f'), ('Save & Run All', 'placeholder file;\nloads your model', '#4c6ef5'),
             ('Submit', 'host reruns all 32 games\non its GPU', '#c96442'), ('Score', 'appears after\nabout 6 hours', '#e0a100')]
    for i, (a, b, c) in enumerate(steps):
        x = .1 + i * 2.4
        ax.add_patch(FancyBboxPatch((x, 1.05), 2.1, 1.1, boxstyle='round,pad=0.02,rounding_size=.12', fc=c, ec='none'))
        ax.text(x + 1.05, 1.9, a, ha='center', va='center', color='white', fontweight='bold', fontsize=11)
        ax.text(x + 1.05, 1.45, b, ha='center', va='center', color='white', fontsize=7.8, linespacing=1.25)
        if i < 4: ax.annotate('', (x + 2.38, 1.6), (x + 2.12, 1.6), arrowprops=dict(arrowstyle='-|>', color=INK, lw=1.6))
    ax.text(0.1, 2.75, 'One turn of the improvement loop', fontsize=13, fontweight='bold', va='center')
    ax.text(0.1, .55, 'Only the last step counts for the leaderboard, and it takes about 6 hours: at most 5 scored tries a day.\nEverything before it is free, so most of the work happens in the cheap steps.', fontsize=10.3, color=MUTED, va='center', linespacing=1.4)
    # (b) game time vs wall time
    bx = fig.add_subplot(gs[1]); bx.axis('off'); bx.set_xlim(0, 12); bx.set_ylim(0, 2)
    bx.text(0.1, 1.7, 'The games are the clock', fontsize=13, fontweight='bold', va='center')
    bx.text(0.1, .8, f'{N_GAMES} games x {EPISODE_S//60} min = {N_GAMES*EPISODE_S/3600:.1f} h of game time is the minimum for one scored run. A model kept loaded\non the GPU for all of it occupies about {N_GAMES*EPISODE_S/3600:.1f} h of GPU time per submission, whatever your agent does.\nThere is no cheap way to score "a few games".', fontsize=10.3, va='center', linespacing=1.4, color=INK)
    # (c) weights memory
    cx = fig.add_subplot(gs[2])
    sizes = [8, 31, 70, 120]; bits = [(16, 'bf16 / fp16', '#c96442'), (8, '8-bit', '#e0a100'), (4, '4-bit', '#2a9d8f')]
    w = .26
    for j, (b, lab, col) in enumerate(bits):
        v = [s * b / 8 for s in sizes]
        cx.bar(np.arange(len(sizes)) + (j - 1) * w, v, w, color=col, label=lab)
        for k, vv in enumerate(v): cx.text(k + (j - 1) * w, vv + 2, f'{vv:.0f}', ha='center', fontsize=8.8)
    cx.axhline(96, color=INK, lw=1.6, ls='--'); cx.text(-.45, 99, 'RTX Pro 6000: 96 GB', ha='left', fontsize=10, fontweight='bold', va='bottom')
    cx.axhline(24, color=MUTED, lw=1.2, ls=':'); cx.text(-.45, 27, 'a 24 GB home GPU', ha='left', fontsize=9.5, color=MUTED, va='bottom')
    cx.set_xticks(range(len(sizes))); cx.set_xticklabels([f'{s}B parameters' for s in sizes], fontsize=11); cx.set_ylabel('weights only, GB'); cx.set_ylim(0, 270); cx.set_xlim(-.5, 3.5)
    cx.set_title('What fits in memory? (weights only: image tokens and the KV cache need more)', loc='left', fontsize=12, fontweight='bold')
    cx.legend(frameon=False, loc='upper center', fontsize=10.5, ncol=3, bbox_to_anchor=(.5, 1.0)); cx.grid(axis='y', alpha=.35, color=RULE)
    plt.show()
cost_figure()

In [ ]:
def profile_figure():
    traits = [('Feedback speed', 2, 'a score arrives about 6 h after you submit, 5 scored tries a day (minutes = 5 stars, 9 h or more = 1 star)'),
              ('GPU appetite', 4, 'vision-language agents use the 96 GB GPU; a CPU-only agent is allowed (<= 9 h)'),
              ('Engineering complexity', 5, 'browser harness, inference server, model serving, a 200 ms clock, 90 s decision cap'),
              ('Generalisation pressure', 5, '28 of 32 games hidden; the private board holds none of the 4 practice games'),
              ('Score noise', 2, 'games are repeatable; the host reports 0.001 to 0.01 of run-to-run change for sampling agents'),
              ('Training data supplied', 1, 'four games and a harness, no dataset; external data and pretrained models are allowed'),
              ('Beginner entry', 3, 'two working starters; but both need the GPU, and local play needs Playwright'),
              ('Domain knowledge', 3, 'games, interfaces and model serving matter more than statistics')]
    fig, ax = plt.subplots(figsize=(12, 5.9)); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(-.4, len(traits) + .6)
    ax.text(0, len(traits) + .2, 'Profile of the competition (1 to 5; our rating, the evidence is from the rules)', fontsize=13, fontweight='bold')
    for i, (n, v, why) in enumerate(traits):
        y = len(traits) - 1 - i
        if i % 2 == 0: ax.add_patch(Rectangle((0, y - .4), 12, .8, color='#f3f0e8', lw=0, zorder=0))
        ax.text(.1, y, n, va='center', fontweight='bold', fontsize=11)
        for k in range(5): ax.add_patch(Circle((3.25 + k * .46, y), .15, fc='#c96442' if k < v else 'white', ec='#c96442', lw=1.3))
        ax.text(5.75, y, textwrap.fill(why, 70), va='center', fontsize=9.3, color=INK, linespacing=1.25)
    plt.show()
profile_figure()

## 10. Who should join, and who might prefer something else
Not every Kaggler gets the same thing out of this competition. The chart gives our honest fit rating per background (full circle = good fit, half = workable, empty = poor), with the reason.

In [ ]:
def fit_figure():
    rows = [('Vision-language agent builders', 2, 'The starters are exactly this; the open problems (latency, memory, memory of past frames) are the contest.'),
            ('Inference / quantisation engineers', 2, 'Reply time is part of the score: faster decoding, smaller models and smart caching directly buy decisions.'),
            ('RL and game-AI researchers', 1, 'Only four games to practise on and none of the hidden 28; value is in generalisation, not in over-training four games.'),
            ('Computer-vision and OCR people', 2, 'Reading scores, menus and buttons from raw frames is unsolved plumbing that every agent needs.'),
            ('ARC-AGI-3 competitors', 2, 'Same question (learn unknown rules fast); different clock. Harness and serving skills transfer well.'),
            ('Students and portfolio builders', 1, 'A research-style problem with MIT-licensed code, a working starter and a clear story. Needs GPU quota to try the starters.'),
            ('Anyone without GPU access', 1, 'CPU-only agents are allowed and local play is free, but the big model starters will not run. Expect a rule-based or small-model agent.'),
            ('Tabular / forecasting specialists', 0, 'No table, no cross-validation, a 6-hour feedback loop. Your toolbox does not apply here.'),
            ('Fast-feedback seekers', 0, 'At most 5 scored tries a day, each about 6 hours. Prefer a Playground or code-golf style contest.')]
    fig, ax = plt.subplots(figsize=(12, 6.4)); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(-.4, len(rows) + .5)
    cmap = {2: '#2a9d8f', 1: '#e0a100', 0: '#c0392b'}; lab = {2: 'good fit', 1: 'workable', 0: 'poor fit'}
    for i, (n, v, why) in enumerate(rows):
        y = len(rows) - 1 - i
        if i % 2 == 0: ax.add_patch(Rectangle((0, y - .42), 12, .84, color='#f3f0e8', lw=0, zorder=0))
        c = cmap[v]; ax.add_patch(Circle((.3, y), .2, fc='white', ec=c, lw=1.8))
        if v == 2: ax.add_patch(Circle((.3, y), .2, fc=c, ec=c))
        if v == 1: ax.add_patch(Wedge((.3, y), .2, 90, 270, fc=c, ec=c))
        ax.text(.7, y + .1, n, va='center', fontsize=10.2, fontweight='bold'); ax.text(.7, y - .22, lab[v], va='center', fontsize=8.8, color=c, fontweight='bold')
        ax.text(5.9, y, textwrap.fill(why, 68), va='center', fontsize=9.1, linespacing=1.25)
    plt.show()
fit_figure()

### If your interests are elsewhere
These competitions were open on Kaggle on 23 November 2026 (deadlines and prizes as listed by Kaggle that day; check each page, they change). Matched to the interest that fits them best:

In [ ]:
def alt_figure():
    rows = [('Quick feedback, tabular', 'Playground Series S6E11', '30 Nov 2026', 'swag'),
            ('Language models: telling models apart', 'Name That LLM', '14 Jan 2027', '$50,000'),
            ('Language-model agents', 'Gemma 4 developer agent', '2 Dec 2026', '$75,000'),
            ('Optimisation and puzzles', 'Santa 2026', '28 Jan 2027', '$50,000'),
            ('Optimisation and geometry', 'Squares in Squares', '18 Jul 2027', '$50,000'),
            ('Science and chemistry', 'Enveda CASMI26 (mass spectra)', '14 Dec 2026', '$50,000'),
            ('Quantum computing', 'Google quantum circuit compilation', '20 Jan 2027', '$150,000'),
            ('Sports analytics', 'NFL Big Data Bowl 2027', '6 Jan 2027', '$100,000'),
            ('Card-game agents', 'Pokemon TCG AI Battle (playground)', '8 Jan 2027', 'swag'),
            ('Very hard reasoning', "Physical Engineering's Last Exam", '5 Jan 2027', '$400,000')]
    fig, ax = plt.subplots(figsize=(12, 5.1)); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(-.2, len(rows) + .7)
    for x, h in [(.1, 'If you like...'), (4.1, 'Competition'), (9.2, 'Deadline'), (11.9, 'Prize')]: ax.text(x, len(rows) + .25, h, fontsize=10.5, color=MUTED, fontweight='bold', ha='right' if h == 'Prize' else 'left')
    for i, (a, b, c, d) in enumerate(rows):
        y = len(rows) - 1 - i
        if i % 2 == 0: ax.add_patch(Rectangle((0, y - .4), 12, .8, color='#f3f0e8', lw=0, zorder=0))
        ax.text(.1, y, a, va='center', fontsize=10.5); ax.text(4.1, y, b, va='center', fontsize=10.8, fontweight='bold'); ax.text(9.2, y, c, va='center', fontsize=10.3); ax.text(11.9, y, d, va='center', fontsize=10.3, ha='right')
    plt.show()
alt_figure()

<div style="border-left:6px solid #2a9d8f;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#2a9d8f">A one-line decision rule</b><br>Join if you enjoy <b>building a system</b> (a model, a harness, a clock-aware loop) more than <b>tuning a number</b>, you can get GPU time, and you are happy with a slow, noisy-looking feedback loop. Skip it, or watch from the side, if you want daily leaderboard moves from a few hours of work.</div>

## 11. Ideas nobody has tested here
These follow from the harness rules and the two starters. They are **hypotheses**: we have not run any of them, we do not know which pay off, and several could cost
more than they give (extra frames mean more image tokens and slower replies). They are listed so that the open directions are visible.

In [ ]:
ideas = [
    ('Act while thinking', 'sequence tool', 'Keep the ship moving or the car steering with a short program while the model plans the next one.'),
    ('Reply with skip', 'skip tool', 'When a program is already doing the right thing, answer skip instead of restarting it every turn.'),
    ('Use the frames you get', 'obs.frames, obs.history', 'Feed two or three recent frames so the model can see direction and speed, not a still picture.'),
    ('Two speeds', 'agent design', 'A cheap reflex policy for fast games, a slow model that sets goals every few seconds.'),
    ('Know when a game restarts', 'obs.elapsed_s', 'The clock drops to near 0 at a new game: reset state, then look for the start screen.'),
    ('Remember within a game', 'agent design', 'Remember which key started the game and which click restarted it, to avoid paying that again.'),
    ('Read the score on screen', 'pixels', 'Some games print their score; reading it could give the agent a signal the starters do not use.'),
    ('Spend the clock deliberately', 'obs.info.remaining_s', 'Explore early, play safe late, or the reverse: a policy on remaining time.'),
]
def ideas_figure():
    fig, ax = plt.subplots(figsize=(12, 6.2)); ax.axis('off'); ax.set_xlim(0, 12); ax.set_ylim(-.6, len(ideas) + .9)
    ax.text(0, len(ideas) + .35, 'Idea', fontweight='bold', fontsize=11, color=MUTED); ax.text(3.5, len(ideas) + .35, 'Harness feature it uses', fontweight='bold', fontsize=11, color=MUTED)
    ax.text(6.4, len(ideas) + .35, 'The thought', fontweight='bold', fontsize=11, color=MUTED); ax.text(11.95, len(ideas) + .35, 'status', fontweight='bold', fontsize=11, color=MUTED, ha='right')
    for i, (a, b, c) in enumerate(ideas):
        y = len(ideas) - 1 - i + .0
        if i % 2 == 0: ax.add_patch(Rectangle((0, y - .4), 12, .8, color='#f3f0e8', lw=0, zorder=0))
        ax.text(0.05, y, a, va='center', fontweight='bold', fontsize=11); ax.text(3.5, y, b, va='center', fontsize=9.2, family='monospace', color='#2a6f66')
        ax.text(6.4, y, textwrap.fill(c, 48), va='center', fontsize=9.0, linespacing=1.25)
        ax.add_patch(FancyBboxPatch((10.75, y - .17), 1.18, .34, boxstyle='round,pad=0,rounding_size=.1', fc='white', ec=RED, lw=1.2)); ax.text(11.34, y, 'untested', ha='center', va='center', fontsize=9, color=RED, fontweight='bold')
    plt.show()
ideas_figure()

<div style="border-left:6px solid #c0392b;background:#f4f6f5;color:#1f2430;padding:12px 18px;border-radius:6px;margin:12px 0;font-size:15px;line-height:1.5"><b style="color:#c0392b">The risk to keep in front of you</b><br>Only <b>4 of 32</b> games are visible, and the final ranking uses the 16 that include <i>none</i> of them. Anything tuned on the four practice games is tuned on a quarter of the public half and none of the private half. A policy that looks great on Asteroid Capitalist and breaks on a menu it has never seen can lose on the private board.</div>

## 12. Limits and credits
- **Sections 1 to 12 never play a game.** Section 14 reports runs made in an earlier version of this notebook, on the four practice games only, for three minutes each and once per agent. They record what the agent does, **not a score** (the practice games' score readers are not shipped), and are short, single samples.
- **Timing numbers** follow the harness text (`protocol.rules()`). The runs in section 14 check them: replies padded to 1.25 s arrive 28 frames (1.4 s) apart, as computed in section 3.
- **Board scores in section 13** are the authors' own statements in their notebook titles and texts on 24 Nov 2026 (plus our submission of the host's starter); we did not re-run them.
- **Engines** are inferred from file names and bundled libraries.
- **Control labels** (section 5) are our reading of the briefs, written in our own words.
- **The 12 hidden public-board games** and the 16 private ones are not published.

**Credits.** Competition by GARC (Generalist Agent Research Collective) with Princeton University as sponsor; competition data and starters under the licence stated on the competition pages (MIT).
The two starter notebooks we describe were published by the host team. The agent in section 15 is the public notebook **0.107 | Qwen3.5-122B-A10B Q4_K_M + Reflex Control**, version 13, by lightningv08, copied unchanged (licence Apache 2.0); the other agents in the section 13 chart and the 0.138 agent measured in section 14 (alexchilton) are the work of their authors (kozykappa, sidhaarthshree, alexchilton). Figures were drawn here with matplotlib; there are no AI-generated images.

**Companion dataset:** `GGA Visual Guide Tables` holds the control-scheme table, the timing and action-space tables used above and the per-run table of section 14 (our own measurements; no competition files).
If this guide helped, an upvote helps others find it.

**This is version 3 and it will keep growing.** If you have an opinion about it, a correction, or a section or analysis you would like to see in the next versions (a particular game or genre, a latency or memory experiment, a comparison with another competition, a different view of the scoring), please say so in the comments here or in the competition discussion topic. Disagreement is welcome, especially on the comparisons and ratings, which are our own reading.

In [ ]:
# Tables behind the figures (also saved to /kaggle/working, and published as the companion dataset)
lat = np.round(np.arange(0.1, 5.01, 0.05), 2)
latency_table = pd.DataFrame({'reply_seconds': lat, 'grid_points_late': delay_grid_points(lat),
                              'action_age_seconds': np.round(delay_grid_points(lat) * GRID_S, 2),
                              'decisions_per_game': np.round(POINTS_PER_GAME / delay_grid_points(lat), 1)})
out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(os.environ.get('LOCAL_OUT', '.'))
out.mkdir(exist_ok=True, parents=True)
latency_table.to_csv(out / 'latency_budget.csv', index=False)
action_space.to_csv(out / 'action_space.csv', index=False)
print('latency_budget.csv', latency_table.shape, '| action_space.csv', action_space.shape)

## 13. The public agents, and how to read 0.291
A few days in, public notebooks score far above the host's own starter (0.093). The chart collects the ones whose scores are visible on the competition's Code tab; the bars are **their** numbers, we did not re-run them.
Two things stand out. Among the Gemma notebooks, **what changed is the agent around the model** (when a reply goes out, how the first screen is handled), not the model. And the top bar is a different kind of entry: it carries **a hand-written controller for each of the four practice games**.

In [ ]:
# Public agents and their board scores, as the authors state them in their notebooks / on the competition's Code tab (25 Nov 2026),
# plus our own submission of the host's starter. Nothing here was re-run by us except the host starter (our submission 58797459).
pub = pd.DataFrame([
    ('Qwen 125B + 4 practice-game\ncontrollers (lightningv08, V13)', 0.291, 'scripted for the practice games', 'A 125B Qwen model for unknown games, plus a hand-written controller for each of the four practice games'),
    ('Start step + no wait\n(alexchilton)',                       0.138, 'host starter + code',  'The host Gemma starter, plus a start step in code and no 1.25 s padding'),
    ('Async planner\n(kozykappa)',                                0.129, 'new agent design',     'Gemma plans 1-3 s of play in a small command language, in a background thread'),
    ('Planner-actor + coach\n(sidhaarthshree)',                   0.113, 'new agent design',     'Fast actor plus a slower reflective coach, motion hints, 0-999 coordinate grid'),
    ('Qwen 125B, one controller\n(lightningv08, V8)',             0.107, 'other model',          'The same Qwen model, with a hand-written controller for one practice game only'),
    ('Host Gemma starter\n(our submission)',                      0.093, 'host starter',         'Unchanged; our own submission 58797459'),
    ('Host Drex Max starter\n(reported by alexchilton)',          0.056, 'host starter',         'Unchanged; the same author measured 0.056 and 0.058 on two runs'),
], columns=['label', 'score', 'kind', 'idea'])
KIND_COL = {'scripted for the practice games': RED, 'host starter + code': '#2a9d8f', 'new agent design': '#4c6ef5', 'other model': '#e0a100', 'host starter': '#9aa0a6'}

fig, ax = plt.subplots(figsize=(11.5, 5.6))
y = np.arange(len(pub))[::-1]
for yi, (_, r) in zip(y, pub.iterrows()):
    ax.plot([0, r.score], [yi, yi], color=KIND_COL[r.kind], lw=3.2, solid_capstyle='round', alpha=.9)
    ax.plot([r.score], [yi], 'o', color=KIND_COL[r.kind], ms=11)
    ax.text(r.score + 0.004, yi, f'{r.score:.3f}', va='center', fontsize=11.5, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(pub.label, fontsize=10)
ax.set_ylim(-0.8, len(pub) - 0.4); ax.set_xlim(0, 0.34); ax.set_xlabel('public leaderboard score (mean of 16 games, each scaled 0 to 1)')
ax.axvline(0.093, color=MUTED, lw=1, ls=':'); ax.text(0.0945, -0.62, 'host Gemma starter', fontsize=9, color=MUTED)
handles = [plt.Line2D([0], [0], marker='o', ls='', color=c, ms=9, label=k) for k, c in KIND_COL.items()]
ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=10)
ax.set_title('Public notebooks on the board (25 Nov 2026)', loc='left', fontsize=12.5, fontweight='bold')
plt.tight_layout(); plt.show()
pub[['label', 'score', 'idea']].assign(label=lambda d: d.label.str.replace('\n', ' ')).to_csv(
    (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(os.environ.get('LOCAL_OUT', '.'))) / 'public_agents_board.csv', index=False)

### Reading 0.291
The public board averages 16 games, **four of which are the practice games** that anyone can play and study; the private board that decides the ranking has 16 other games, none of them practice games. An agent that scripts the four practice games therefore moves the public board a lot and the private board not at all.
The chart does the arithmetic: the board alone cannot split 0.291 into "the controllers" and "the general agent", but it bounds the general agent's score, which is what a new game will see.

In [ ]:
# What does 0.291 say about the private board? The public board is 16 games: the 4 practice games and 12 hidden ones.
# The private board is 16 other hidden games, none of them a practice game. If the public mean is M, the general agent
# scores g on a hidden game and the four controllers average p on the practice games:   M = (4 p + 12 g) / 16.
M, N_PRACTICE, N_HIDDEN = 0.291, 4, 12
g = np.linspace((M * 16 - N_PRACTICE) / N_HIDDEN, M, 200)       # p can be at most 1, and g cannot exceed M (p = g)
p = (M * 16 - N_HIDDEN * g) / N_PRACTICE
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={'width_ratios': [1, 1.35]})
ax = axs[0]; ax.axis('off'); ax.set_xlim(0, 8.2); ax.set_ylim(-0.4, 6.3)
def board(y0, title, cells):
    ax.text(0, y0 + 2.1, title, fontsize=11, fontweight='bold', va='bottom')
    for i, c in enumerate(cells):
        r, k = divmod(i, 8)
        ax.add_patch(Rectangle((k * 1.0, y0 + 1.0 - r * 1.05), 0.85, 0.85, color=c, lw=0))
board(3.7, 'Public board: 16 games', [RED] * 4 + ['#c9c5ba'] * 12)
board(0.7, 'Private board: 16 other games', ['#c9c5ba'] * 16)
ax.text(0, -0.3, 'red = a practice game with its own hand-written controller;  grey = a hidden game', fontsize=9, color=MUTED)
ax = axs[1]
ax.plot(g, p, color=INK, lw=2.4)
ax.fill_between(g, 0, p, color=RED, alpha=.08)
ax.axvline(M, color=MUTED, lw=1, ls=':')
ax.set_xlim(0, 0.31); ax.set_ylim(0, 1.05)
ax.set_xlabel('what the general agent scores on a hidden game, g\n(this is also about what it would score on the private board)')
ax.set_ylabel('average of the four practice-game controllers, p')
g0 = 0.107; p0 = (M * 16 - N_HIDDEN * g0) / N_PRACTICE
ax.plot([g0], [p0], 'o', color=RED, ms=10); ax.annotate(f'if g = {g0:.3f} (the score of the same notebook before it\nhad all four controllers), p must be {p0:.2f}', (g0, p0), (0.012, 0.28), fontsize=9.5, arrowprops=dict(arrowstyle='-', color=MUTED))
ax.text(M - 0.004, 0.06, f'g = {M}: the\ncontrollers add\nnothing', fontsize=9, ha='right', color=MUTED)
ax.set_title('0.291 = (4 p + 12 g) / 16', loc='left', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()
box('How to read 0.291',
    f"<ul style='margin:6px 0 0 18px'><li>The author says it plainly: <i>the 12 hidden public games use the general agent</i>; the four practice games each have a controller written for that one game and selected by its id.</li>"
    f"<li>The private board contains <b>none</b> of the four practice games, so those controllers earn nothing there. What carries over is the general agent, whose private score is about <b>g</b> in the chart, somewhere between {g.min():.2f} and {M:.2f}, not 0.291.</li>"
    f"<li>Both numbers are worth knowing: <b>0.291</b> is the public-board record, and <b>g</b> is what a new game will see. The competition ranks on the private board.</li></ul>", '#c0392b')

## 14. Real game statistics: what two agents do on the four practice games
Measured on the competition GPU in the previous version of this guide (version 7): the host's Gemma starter (replies padded to 1.25 s) and the 0.138 notebook (no wait, start step), each on the four practice games for 180 s, one run each. The numbers are in the companion dataset (`real_game_runs.csv`) and are read from there, so this section runs without a GPU.
They record what the **agent does**: how many decision points it answers, how long a reply takes, how often the picture changes, how long it sits on a still screen. **They are not scores**: the game's own score reader (`Game.get_score()`) returns nothing for the practice games, because the readers are not part of the competition files.

In [ ]:
RUNS_CSV = next(INPUT.rglob('real_game_runs.csv'), None)
if RUNS_CSV is None:
    print('real_game_runs.csv is not in the attached dataset.')
else:
    runs = pd.read_csv(RUNS_CSV)
    AG = ['host starter', 'start step, no wait']
    AG_COL = {AG[0]: '#9aa0a6', AG[1]: '#2a9d8f'}
    BENCH_GAMES = ['asteroid_capitalist', 'blarglike', 'cavebots', 'off_line']
    MAX_PER_MIN = 60 / 0.2                                   # one decision point every 200 ms
    G = {a: runs[runs.agent == a].set_index('game').reindex(BENCH_GAMES) for a in AG}

    def grouped(ax, col, title, ylabel, fmt, ylim=None):
        x = np.arange(len(BENCH_GAMES)); w = 0.36
        for k, a in enumerate(AG):
            vals = G[a][col].astype(float).values
            ax.bar(x + (k - .5) * w, vals, w, color=AG_COL[a], label=a)
            for xi, v in zip(x + (k - .5) * w, vals):
                if np.isfinite(v): ax.text(xi, v + (ylim or np.nanmax(vals) or 1) * .015, fmt.format(v), ha='center', fontsize=9)
        ax.set_xticks(x); ax.set_xticklabels([GAME_NAME[g].replace(' ', '\n') for g in BENCH_GAMES], fontsize=9.5)
        ax.set_ylabel(ylabel); ax.set_title(title, loc='left', fontsize=11.5, fontweight='bold')
        if ylim: ax.set_ylim(0, ylim)

    fig, axs = plt.subplots(2, 2, figsize=(12.5, 7.4))
    grouped(axs[0, 0], 'decisions_per_minute', 'Decision points the agent answered', 'per minute (300 possible)', '{:.0f}', ylim=330)
    axs[0, 0].axhline(MAX_PER_MIN, color=RED, lw=1, ls=':'); axs[0, 0].legend(frameon=False, fontsize=9, loc='upper left')
    grouped(axs[0, 1], 'changed_share', 'Decision points where the picture changed', 'share of decision points', '{:.0%}', ylim=1.12)
    grouped(axs[1, 0], 'still_longest_s', 'Longest stretch with an unchanged picture', 'seconds of game time', '{:.0f} s')
    ax = axs[1, 1]
    x = np.arange(len(BENCH_GAMES)); vals = G[AG[1]]['menu_share'].astype(float).values
    ax.bar(x, vals, .5, color=AG_COL[AG[1]])
    for xi, v in zip(x, vals): ax.text(xi, v + .015, f'{v:.0%}', ha='center', fontsize=9.5)
    ax.set_xticks(x); ax.set_xticklabels([GAME_NAME[g].replace(' ', '\n') for g in BENCH_GAMES], fontsize=9.5)
    ax.set_ylim(0, max(0.5, float(np.nanmax(vals)) * 1.25)); ax.set_ylabel('share of the run')
    ax.set_title('Time the start step took for the first screen', loc='left', fontsize=11.5, fontweight='bold')
    plt.tight_layout(); plt.show()

    show = runs[['agent', 'game', 'turns', 'decisions_per_minute', 'reply_median_s', 'reply_p95_s', 'frame_gap_median', 'changed_share', 'still_longest_s', 'menu_share', 'refused']].copy()
    show.columns = ['agent', 'game', 'turns', 'answered per min', 'median reply s', 'p95 reply s', 'median frames between screenshots', 'picture changed', 'longest still s', 'share on first screen', 'refused']
    display(show.round(2).style.hide(axis='index').format(precision=2))
    a0, a1 = AG
    d0, d1 = runs[runs.agent == a0].decisions_per_minute.mean() / MAX_PER_MIN, runs[runs.agent == a1].decisions_per_minute.mean() / MAX_PER_MIN
    f0, f1 = runs[runs.agent == a0].frame_gap_median.median(), runs[runs.agent == a1].frame_gap_median.median()
    BENCH_SECONDS_MEASURED = 180
    stuck = [g for g in BENCH_GAMES if all(G[a].loc[g, 'still_longest_s'] > 60 for a in AG)]
    longest = ', '.join('%.0f s' % G[AG[1]].loc[g, 'still_longest_s'] for g in stuck)
    stuck_txt = (f"With <i>both</i> agents the picture stayed unchanged for more than a minute in <b>{', '.join(GAME_NAME[g] for g in stuck)}</b> "
                 f"(longest still stretch {longest} of {BENCH_SECONDS_MEASURED} s). Neither starter gets through whatever blocks those games; "
                 f"a controller written for the game does, which is why the practice games are the easy place to gain on the public board."
                 if stuck else 'No game stayed still for more than a minute with both agents.')
    box('What these runs say, and what they do not',
        f"<ul style='margin:6px 0 0 18px'>"
        f"<li><b>Answering.</b> The host starter answered <b>{d0:.0%}</b> of the decision points (a screenshot every {f0:.0f} frames, one reply every {f0 / FPS:.1f} s); the agent with no wait answered <b>{d1:.0%}</b> (a screenshot every {f1:.0f} frames in the median). "
        f"The median reply took {runs[runs.agent == a0].reply_median_s.median():.2f} s against {runs[runs.agent == a1].reply_median_s.median():.2f} s.</li>"
        f"<li><b>Stuck screens.</b> {stuck_txt}</li>"
        f"<li><b>No scores.</b> The game's own score <i>cannot</i> be read in the practice games (<code>Game.get_score()</code> returns <code>None</code>: the score readers are not in the competition files). "
        f"Nothing here says which agent scores higher; the leaderboard does.</li>"
        f"<li><b>Limits.</b> One run per pair and game, a fixed seed, {BENCH_SECONDS_MEASURED} s instead of 600 s, only the 4 practice games. These are the two Gemma 4 31B agents of the previous version of this guide (the host starter and the 0.138 notebook); the agent this notebook submits now is a different one (section 15).</li></ul>", '#2a9d8f')

## 15. The agent this notebook submits: Qwen 125B + four practice-game controllers (board 0.291)
Everything below is the public notebook **"0.107 | Qwen3.5-122B-A10B Q4_K_M + Reflex Control"** by **lightningv08**, **version 13** (the version behind its best board score, 0.291), copied without changes except for the notebook's own text. Its title still shows the older 0.107.
It runs a 125B-parameter Qwen model (4-bit weights, served with SGLang) as the general player, and routes each of the four practice games to a CPU controller written for that game (Off Line, Asteroid Capitalist, Cavebots, Blarglike); see the `Agent` class near the end. The author's own text says the 12 hidden public games use the general agent.
**All credit for this code goes to its authors and to the sources it names:** parthenos (QwenFN, SGLang loader), Malyshev Danil (planner/actor), dfranzen (the Pennyroyal runtime and the Intel AutoRound weights), Carson Rodrigues (offline Python), Black Cat Reflex. Licence: Apache 2.0.
It needs the RTX Pro 6000, the competition data, the Qwen model and two public datasets attached; the setup cell starts the model server in the background.

### Offline runtime

The 125B mixture of experts uses 4-bit AutoRound weights, an SM120 SGLang build, an FP8 KV cache and a NextN draft model.

In [ ]:
import io, os, sys, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
root = Path('/kaggle/input')
harness = next((p for p in [root/'competitions/generalist-gaming-agent', root/'generalist-gaming-agent']
                if (p/'gga/protocol.py').is_file()), None)
if harness is None:
    harness = next(root.rglob('gga/protocol.py')).parents[1]
sys.path.insert(0, str(harness))
os.environ['GGA_ROOT'] = str(harness)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
rerun = 'KAGGLE_IS_COMPETITION_RERUN' in os.environ
print('Competition harness', harness.name, '| scoring run', rerun)

In [ ]:
# Pennyroyal setup adapted from dfranzen and parthenos, linked in Sources.
import time
from pathlib import Path
boot_time=time.monotonic()
WORKING_DIR=Path('/kaggle/working')
WHEELHOUSE_DIR=next(p.parent for p in root.rglob('requirements.lock') if 'pennyroyal-v253' in str(p))
FLASH_MODEL_DIR=next(p.parent for p in root.rglob('model.safetensors.index.json') if 'intel-qwen3.8-flash-next-w4a16' in str(p))
FLASH_DRAFT_DIR=next(p.parent for p in root.rglob('model.safetensors.index.json') if 'flash-next-drafter' in str(p))
SERVED_MODEL_NAME='flashnext'
SERVED_MODEL_PORT=8128
FLASH_URL=f'http://127.0.0.1:{SERVED_MODEL_PORT}'
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error, threading, signal
import contextlib, copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

PREFIX = "/tmp/sgl-intel"
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=8 * 1024,           # prompts here are ~1-4k tokens
    MEMFRAC=0.94,
    MAXREQ=2,
    CUDAGRAPH_MAXBS=2,
    MAMBA_CACHE=16,          # 8 slots per request (the ARC config used 6: 60 for 10)
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=FLASH_DRAFT_DIR is not None,   # MTP drafter: ~2-3x decode speed with greedy decoding
    FRSPEC=True,
    AUTOTUNE=False,
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view




def precache(*paths, times=1, delay=0, threads=16) -> None:
    if delay: time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')



def find_python312():
    import tarfile
    archive=next(Path('/kaggle/input').rglob('cpython-3.12.*-x86_64-linux-gnu-install_only.tgz.bin'))
    expected=(archive.parent/'SHA256SUM.txt').read_text().split()[0]
    if sha256(archive)!=expected:
        raise ValueError('Portable Python checksum mismatch')
    dest=Path('/tmp/gga-python312')
    dest.mkdir(exist_ok=True)
    with tarfile.open(archive,'r:gz') as tf:
        tf.extractall(dest,filter='data')
    exe=dest/'python/bin/python3.12'
    probe=subprocess.check_output([str(exe),'-c','import sys,sysconfig,pathlib;print(sys.version_info[:2],pathlib.Path(sysconfig.get_paths()["include"],"Python.h").is_file())'],text=True)
    assert '(3, 12) True' in probe,probe
    return str(exe)

def _py_version(exe):
    version=subprocess.check_output([str(exe),'-c','import sys;print(*sys.version_info[:2])'],text=True).split()
    return tuple(map(int,version))

PY312=find_python312()
def _prepare_flashnext(MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR):
    # ---- validate notebook inputs / identify the offline bundle ----
    MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
    assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
    assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
    assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
    # Refuse to start a second server on an occupied port; do not kill other jobs.
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
            raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
    Path(PREFIX).mkdir(parents=True, exist_ok=True)
    Path(LOG).parent.mkdir(parents=True, exist_ok=True)
    WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
    LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
    sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
    assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
    wheel = sglang_wheels[0]
    install_marker = Path(PREFIX) / "installed-bundle.txt"

    # ---- one-time offline install, isolated from the old mratsim environment ----
    install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
    install_env.pop("PYTHONPATH", None)
    if not Path(SGLANG).exists() or not install_marker.is_file() or (_py_version(PYTHON) or (0, 0))[:2] != (3, 12):
        print("=== installing Pennyroyal (offline) ===")
        # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
        uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
        assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
        uv_binary = Path(PREFIX) / "bin/uv"
        uv_binary.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(uv_wheels[0]) as archive:
            members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
            assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
            binary = archive.read(members[0])
            assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
            uv_binary.write_bytes(binary)
        uv_binary.chmod(0o755)
        uv = [str(uv_binary)]
        if Path(PYTHON).exists() and (_py_version(PYTHON) or (0, 0))[:2] != (3, 12):
            shutil.rmtree(VENV, ignore_errors=True)   # left over from an attempt with the wrong interpreter
        if not Path(PYTHON).exists():
            run(uv + ["venv", "--python", PY312, VENV], env=install_env)
        install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
        run(install + ["-r", str(LOCK)], env=install_env)
        run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
        install_marker.write_text('install')
    else:
        print(f"Matching offline bundle already installed at {VENV}")

    # ---- bundled CUDA toolkit and linker symlinks ----
    cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
    if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
        raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
    CUDA_HOME = str(cu13[0])
    libdir = Path(CUDA_HOME) / "lib"
    lib64 = Path(CUDA_HOME) / "lib64"
    if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
    for so in sorted(libdir.glob("*.so.*")):
        link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
        if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
    assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
    if not (libdir / "libcuda.so").exists():
        candidates = []
        for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
        if not candidates:
            code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
            maps = run([PYTHON, "-c", code], env=install_env).stdout
            candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
        if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
        link = libdir / "libcuda.so"
        if link.is_symlink(): link.unlink()
        link.symlink_to(candidates[0].resolve())
    CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
    assert CXX, "Host C++ compiler not found"
    CC = CXX.replace("g++", "gcc")
    assert shutil.which(CC), f"Matching C compiler missing: {CC}"

    # ---- runtime environment ----
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    cache = f"{PREFIX}/cache"
    for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
        Path(cache, name).mkdir(parents=True, exist_ok=True)
    env.update({
        "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
        "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
        "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
        "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
        "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
        "TORCH_CUDA_ARCH_LIST": "12.0",
        "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
        "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
        "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
        "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
        "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
        "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
        "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
        "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
        "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
        "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
        "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
        "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
        "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
    })
    run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
    print("Host compiler:", CXX)
    # Report the optional later caching patch. This cell does not modify wheel code.
    run([PYTHON, "-c", "import sglang, torch; "
         "print('sglang', sglang.__version__, '| torch', torch.__version__); "
         "assert torch.cuda.is_available(), 'CUDA unavailable'; "
         "print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))"],
        env=env)

    # ---- validate target and create the read-only draft's SGLang view ----
    target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
    q = target_cfg.get("quantization_config", {})
    assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
    _, target_shards = indexed_shards(MODEL_DIR)
    print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
    DRAFT_VIEW = None
    if CFG["SPEC"]:
        assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
        DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

    # ---- assemble Pennyroyal / AutoRound launch arguments ----
    graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
    graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
    args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
        "--model-loader-extra-config", '{"enable_multithread_load":false}',
        "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
        "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
        "--dtype", "bfloat16", "--quantization", "auto-round",
        "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
        "--context-length", str(CFG["CTX"]), "--page-size", "64",
        "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
        "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
        "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
        "--cuda-graph-bs-decode", *map(str, graph_bs),
        "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
        "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
        "--mamba-backend", CFG["LINEAR_BACKEND"],
        "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
        "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
        "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
        "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
        "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
        "--default-chat-template-kwargs", '{"preserve_thinking":true}',
        "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
        "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
    if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
    if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
    tmpl = Path(MODEL_DIR, "chat_template.jinja")
    if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
    if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
    if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
    if CFG["SPEC"]:
        args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
            "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
            "--speculative-draft-model-path", str(DRAFT_VIEW),
            "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
            "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
            "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
            "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
        if CFG["FRSPEC"]:
            tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
            if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
            assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
            assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
            args += ["--speculative-token-map", str(tok)]


    return args, env


def _flash_healthy(timeout=2.0):
    try:
        with urllib.request.urlopen(f"{FLASH_URL}/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


FLASH_PROC = None
FLASH_ARGS = FLASH_ENV = None


def launch_flash():
    """Start (or restart) the SGLang server process in the background; returns the Popen."""
    global FLASH_PROC
    with open(LOG, "ab", buffering=0) as logf:
        FLASH_PROC = subprocess.Popen(FLASH_ARGS, env=FLASH_ENV, stdout=logf, stderr=subprocess.STDOUT,
                                      start_new_session=True)
    print(f"SGLang pid {FLASH_PROC.pid} -> {LOG} (launched at {time.monotonic() - boot_time:.0f}s)")
    return FLASH_PROC


def show_log_tail(n=6000):
    with contextlib.suppress(Exception):
        with open(LOG, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - n))
            print(f.read().decode(errors="replace"))


if _flash_healthy():
    print("Flash-Next server already running; reusing it")
else:
    precache(str(WHEELHOUSE_DIR))   # wheels first (the install reads them), weights stream in meanwhile
    FLASH_ARGS, FLASH_ENV = _prepare_flashnext(str(FLASH_MODEL_DIR), str(FLASH_DRAFT_DIR or ""), str(WHEELHOUSE_DIR))
    threading.Thread(target=precache, args=(str(FLASH_MODEL_DIR), str(FLASH_DRAFT_DIR or FLASH_MODEL_DIR)),
                     kwargs={"threads": 3}, daemon=True).start()
    print("\n" + shlex.join(FLASH_ARGS) + "\n")
    launch_flash()

FLASH_READY = threading.Event()
def wait_for_model():
    while not _flash_healthy(timeout=2):
        if FLASH_PROC is not None and FLASH_PROC.poll() is not None:
            show_log_tail()
            raise RuntimeError('Flash-Next exited during startup')
        if not rerun and time.monotonic() - boot_time > 1110:
            show_log_tail()
            raise TimeoutError('Flash-Next startup exceeded 1110 seconds')
        time.sleep(1)
    FLASH_READY.set()
    print('Model ready',round(time.monotonic()-boot_time,2),'s', flush=True)

if rerun:
    threading.Thread(target=wait_for_model, daemon=True).start()
    # Allow most of the startup window for weight loading before any game clock starts.
    FLASH_READY.wait(timeout=max(0, 720 - (time.monotonic() - boot_time)))
else:
    wait_for_model()
import base64

### Visual planner and actor

The general policy and multimodal client are adapted from [Malyshev Danil, Smart Unstick V1](https://www.kaggle.com/code/malyshevdanil/gga-brief-aware-planner-actor-smart-unstick?scriptVersionId=373356874), building on [parthenos, QwenFN](https://www.kaggle.com/code/nihilisticneuralnet/gga-qwenfn-agent). Changes include the offline runtime, immediate dispatch after inference, live input feedback, game routing, the practice game controllers, and the diagnostic below. Agents receive screenshots, the supplied game brief and input feedback.

In [ ]:
class FlashNextBackend:
    """Qwen3.8-Flash-Next (125B MoE, 6B active) behind SGLang's OpenAI-compatible API; thinking off, greedy."""
    name = 'flashnext'
    # 6B active params + MTP drafting -> far faster replies than a dense 31B: more decisions, more planning
    timing = dict(PLAN_EVERY_S=20.0, PLAN_MIN_GAP_S=6.0,
                  ACT_MAX_TOKENS=56, PLAN_MAX_TOKENS=260)

    def __init__(self, url, model):
        self.url, self.model = url.rstrip('/'), model

    @staticmethod
    def _data_url(im):
        buf = io.BytesIO()
        im.save(buf, format='JPEG', quality=90)
        return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

    def generate(self, system, images, text, prefill, max_new, mode, budget=None):
        content = [{'type': 'image_url', 'image_url': {'url': self._data_url(im)}} for im in images]
        content.append({'type': 'text', 'text': text})
        body = {
            'model': self.model,
            'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': content}],
            'temperature': 0.0, 'top_p': 1.0, 'top_k': 1, 'max_tokens': int(max_new),
            'chat_template_kwargs': {'enable_thinking': False},
            'skip_special_tokens': True,
        }
        if mode == 'line':
            body['stop'] = ['\n']
        req = urllib.request.Request(f'{self.url}/v1/chat/completions', data=json.dumps(body).encode(),
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=60) as r:
            msg = json.loads(r.read())['choices'][0]['message']
        out = (msg.get('content') or msg.get('reasoning_content') or '').strip()
        out = re.sub(r'(?s)<think>.*?</think>', '', out).strip()
        if prefill and not out.lstrip().upper().startswith(prefill.rstrip(':').upper()):
            out = prefill + ' ' + out
        return out

class Backend(FlashNextBackend):
    last_reply = ''
    last_images = 0

    def ready(self):
        return FLASH_READY.is_set() and (FLASH_PROC is None or FLASH_PROC.poll() is None)

    def generate(self, *args, **kwargs):
        self.last_reply = ''
        self.last_images = len(args[1])
        try:
            self.last_reply = super().generate(*args, **kwargs)
        except (OSError, RuntimeError, ValueError, TimeoutError) as error:
            print('Model response unavailable:', type(error).__name__, flush=True)
        return self.last_reply

MANAGER = Backend(FLASH_URL, SERVED_MODEL_NAME)

In [ ]:
def describe_inputs(obs):
    remaining = float(getattr(obs, 'program_left_s', 0) or 0)
    held = getattr(obs, 'held', None) or {}
    keys = list(held.get('keys') or [])
    buttons = list(held.get('buttons') or [])
    parts = []
    if remaining > 0 or keys or buttons:
        controls = ', '.join(keys + [str(b) + ' mouse' for b in buttons]) or 'no key or button at this instant'
        parts.append(f'Current input: {controls}; {remaining:.1f} game seconds remain in its program.')
    previous = getattr(obs, 'last_action', None) or {}
    latency = previous.get('wall_latency_s')
    if isinstance(latency, (float, int)) and latency >= .2:
        parts.append(f'The previous reply took {latency:.1f} real seconds to arrive; allow for moving targets.')
    return ' '.join(parts)

import contextlib, copy, dataclasses, io, re, time, hashlib
from collections import deque

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from gga.protocol import Action

try:
    from gga.protocol import NAMED_KEYS as _NAMED_KEYS
    NAMED_KEYS = {str(k) for k in _NAMED_KEYS}
except Exception:
    NAMED_KEYS = set()
NAMED_LOWER = {k.lower(): k for k in NAMED_KEYS}

# ---- timing / budget knobs ---------------------------------------------------------------------------
# reply timing and token limits live with the backend (MANAGER.timing)
def T(key):
    return MANAGER.timing[key]


TRAIL_HOLD_MS = 1600     # a final open-ended hold lasts this long (covers the next think time, then releases)
MAX_PROGRAM_MS = 5000
IMG_SIDE = 1024          # screenshot long side handed to the processor
FROZEN_EPS = 0.004       # mean |diff| (0..1) under which the screen counts as unchanged
BIG_CHANGE = 0.22        # scene change (level / game over / menu) -> re-plan
UNSTICK_AFTER = 6        # frozen replies before scripted probes are interleaved
GRID_OVERLAY = True      # faint 0-999 axis grid drawn on the screenshot (axis-grid grounding scaffold)

COMMANDS = '''Commands (coordinates are on a 0-999 grid over the screenshot: x 0=left..999=right, y 0=top..999=bottom; the faint magenta lines and labels mark every 100 units; MS = milliseconds):
  click X Y               left click
  rclick X Y              right click
  clicks X Y N            N fast left clicks on one spot (N up to 40, about 16 per second)
  move X Y [MS]           move the pointer there (hover / aim), then wait MS
  drag X1 Y1 X2 Y2        press at X1 Y1, drag to X2 Y2, release
  tap KEY [N]             press KEY once (or N times)
  hold KEYS [MS]          hold keys together for MS, join keys with + (example: hold ArrowRight+Space 400)
  hold KEYS+Mouse X Y [MS]  also hold the left mouse button with the pointer at X Y (aim and fire)
  continue                keep the current input program running (use alone)
  wait MS                 replace the active input with a neutral wait
  type TEXT               type letters
Keys: single characters (w a s d r 1 2 ...) or ArrowLeft ArrowRight ArrowUp ArrowDown Space Enter Escape Shift Tab Backspace.
Chain commands with ";" to make a short program that runs in order, e.g.
  hold ArrowRight 500; hold ArrowRight+ArrowUp 300; hold ArrowRight
  clicks 480 520 15; click 850 200
  hold w+Mouse 610 340 600; hold d+Mouse 650 330
A hold without MS at the end lasts until your next reply. The game keeps running until your reply arrives. A new input replaces the running program; continue preserves it.'''

SYSTEM_ACT = f'''You are an expert video-game player controlling a real-time browser game from screenshots. The game keeps running while you think, so act decisively and keep inputs going. The score counts in-game progress (points, money, levels, distance, ...); doing nothing scores 0.

{COMMANDS}

How to play well:
- Follow the brief, the controls, and the NOTES from your coach.
- If a title, menu, pause, level-complete or game-over screen shows, get back into play: click its button (Play, Start, Retry, Continue, OK, Next) or press the key it names (often Enter or Space).
- Click exactly on the centre of the buttons and targets you see.
- If recent inputs caused no visible change, do something different.
- The magenta grid lines are a coordinate aid, not part of the game.
Reply with ONE line of commands and nothing else.'''

SYSTEM_PLAN = f'''You are an expert video-game player and coach for a real-time browser game you have never seen. You get screenshots from the last seconds (oldest first, the last one is NOW), the brief, your previous notes, and a log of recent inputs with how much the screen changed after each. Work out how this game is played and how to score more, then give the next inputs. The score counts in-game progress (points, money, levels, distance, ...); doing nothing scores 0.

{COMMANDS}

Tactics by kind of game:
- Clicker / idle / tycoon / management: click the main money source fast (clicks), buy upgrades and producers whenever affordable, keep buying.
- Platformer / runner: move toward the goal (usually right), jump over gaps, hazards and enemies, collect items.
- Shooter / arena / space: keep moving, aim at enemies and fire continuously, dodge bullets.
- Puzzle / board / match / line-drawing: understand the rule, make deliberate moves that make progress, never repeat moves that did nothing.
- Racing / driving: hold accelerate, steer to stay on the track.
- Roguelike / adventure / exploration: explore, fight, pick up items, take exits or stairs to progress, avoid dying.
- Sports / physics / rhythm / typing: time inputs to what the screen shows.
- Title / menu / game-over screens: click Play/Start/Retry or press the key shown (Enter, Space).

Reflect like a coach reviewing a replay: if the score stopped rising, a life was lost or the game ended, say in LEARNED what caused it and how to avoid it; keep what worked, drop what did not. Ignore the magenta grid lines, they are not part of the game.

Reply in exactly this format, one short line each:
SCREEN: what is on screen now
SCORE: the score, money or level shown, or ?
LEARNED: what the inputs did and did not do so far
PLAN: the best concrete tactic for the next 30 seconds (where to click, which keys, what to avoid)
ACT: one line of commands to run now'''

# ---- key normalisation -------------------------------------------------------------------------------
_ALIAS = {
    'space': 'Space', 'spacebar': 'Space', 'enter': 'Enter', 'return': 'Enter', 'esc': 'Escape',
    'escape': 'Escape', 'tab': 'Tab', 'shift': 'Shift', 'shiftleft': 'Shift', 'lshift': 'Shift',
    'ctrl': 'Control', 'control': 'Control', 'controlleft': 'Control', 'alt': 'Alt', 'backspace': 'Backspace',
    'delete': 'Delete', 'del': 'Delete', 'up': 'ArrowUp', 'arrowup': 'ArrowUp', 'uparrow': 'ArrowUp',
    'down': 'ArrowDown', 'arrowdown': 'ArrowDown', 'downarrow': 'ArrowDown', 'left': 'ArrowLeft',
    'arrowleft': 'ArrowLeft', 'leftarrow': 'ArrowLeft', 'right': 'ArrowRight', 'arrowright': 'ArrowRight',
    'rightarrow': 'ArrowRight', 'pageup': 'PageUp', 'pagedown': 'PageDown', 'home': 'Home', 'end': 'End',
    'insert': 'Insert', '↑': 'ArrowUp', '↓': 'ArrowDown', '←': 'ArrowLeft', '→': 'ArrowRight',
}
_LMB = {'mouse', 'lmb', 'mouseleft', 'leftmouse', 'leftclick', 'mouse1', 'leftbutton', 'lclick'}
_RMB = {'rmb', 'mouseright', 'rightmouse', 'rightclick', 'mouse2', 'rightbutton'}
_NUM = re.compile(r'^-?\d+(?:\.\d+)?$')


def norm_key(tok):
    t = tok.strip().strip('"\'`.,:[]{}()')
    if not t:
        return None
    low = t.lower()
    if low in _ALIAS:
        k = _ALIAS[low]
    elif len(t) == 1:
        k = low if t.isalpha() else t
    elif low in NAMED_LOWER:
        k = NAMED_LOWER[low]
    elif re.fullmatch(r'f\d{1,2}', low):
        k = low.upper()
    elif re.fullmatch(r'key[a-z]', low):
        k = low[3]
    elif re.fullmatch(r'digit\d', low):
        k = low[5]
    else:
        return None
    if NAMED_KEYS and len(k) > 1 and k not in NAMED_KEYS:
        return NAMED_LOWER.get(k.lower())
    return k


def _clamp(v, lo, hi):
    return max(lo, min(hi, v))


# ---- command language: parse -> ops -----------------------------------------------------------------
_VERBS = {'click', 'lclick', 'rclick', 'rightclick', 'dblclick', 'doubleclick', 'clicks', 'move', 'hover', 'aim',
          'drag', 'swipe', 'tap', 'press', 'key', 'hold', 'keys', 'wait', 'sleep', 'idle', 'type', 'skip', 'noop', 'continue'}


def _pick_line(text):
    text = text.replace('`', ' ').replace('*', ' ')
    m = re.search(r'ACT\s*:\s*([^\n]*)', text)
    if m and m.group(1).strip():
        return m.group(1)
    for line in text.splitlines():
        words = line.replace(';', ' ').split()
        if words and words[0].lower().rstrip(':') in _VERBS:
            return line
    return text.splitlines()[0] if text.strip() else ''


def parse_program(text):
    ops = []
    for part in re.split(r'[;|]', _pick_line(text)):
        toks = re.sub(r'[,()=]', ' ', part).split()
        if not toks:
            continue
        verb = toks[0].lower().rstrip(':')
        args = toks[1:]
        nums = [float(a) for a in args if _NUM.match(a)]
        words = [a for a in args if not _NUM.match(a)]
        xy = lambda i=0: (_clamp(int(nums[i]), 0, 999), _clamp(int(nums[i + 1]), 0, 999))
        try:
            if verb in ('click', 'lclick') and len(nums) >= 2:
                ops.append({'op': 'click', 'xy': xy(), 'b': 'left'})
            elif verb in ('rclick', 'rightclick') and len(nums) >= 2:
                ops.append({'op': 'click', 'xy': xy(), 'b': 'right'})
            elif verb in ('dblclick', 'doubleclick') and len(nums) >= 2:
                ops.append({'op': 'clicks', 'xy': xy(), 'n': 2})
            elif verb == 'clicks' and len(nums) >= 2:
                n = int(nums[2]) if len(nums) >= 3 else 10
                ops.append({'op': 'clicks', 'xy': xy(), 'n': _clamp(n, 1, 40)})
            elif verb in ('move', 'hover', 'aim') and len(nums) >= 2:
                ms = int(nums[2]) if len(nums) >= 3 else 0
                ops.append({'op': 'move', 'xy': xy(), 'ms': _clamp(ms, 0, 3000)})
            elif verb in ('drag', 'swipe') and len(nums) >= 4:
                ops.append({'op': 'drag', 'xy': xy(0), 'xy2': xy(2)})
            elif verb in ('tap', 'press', 'key') and words:
                keys = [norm_key(k) for w in words for k in w.split('+')]
                keys = [k for k in keys if k]
                n = _clamp(int(nums[0]), 1, 20) if nums else 1
                if len(keys) == 1:
                    ops.append({'op': 'tap', 'key': keys[0], 'n': n})
                elif keys:
                    ops.append({'op': 'hold', 'keys': keys, 'buttons': [], 'xy': None, 'ms': 120})
            elif verb in ('hold', 'keys') or (verb not in _VERBS and norm_key(toks[0]) and len(toks[0]) > 1):
                if verb not in ('hold', 'keys'):
                    words = [toks[0]] + words
                keys, buttons = [], []
                for w in words:
                    for k in w.split('+'):
                        lk = k.lower().strip()
                        if lk in _LMB:
                            buttons.append('left')
                        elif lk in _RMB:
                            buttons.append('right')
                        else:
                            nk = norm_key(k)
                            if nk and nk not in keys:
                                keys.append(nk)
                pos, ms = None, None
                if len(nums) >= 3:
                    pos, ms = xy(), int(nums[2])
                elif len(nums) == 2:
                    pos = xy()
                elif len(nums) == 1:
                    ms = int(nums[0])
                if keys or buttons:
                    ops.append({'op': 'hold', 'keys': keys, 'buttons': sorted(set(buttons)), 'xy': pos,
                                'ms': None if ms is None else _clamp(ms, 0, 3000)})
            elif verb in ('wait', 'sleep', 'idle'):
                ops.append({'op': 'wait', 'ms': _clamp(int(nums[0]) if nums else 300, 0, 3000)})
            elif verb == 'type':
                txt = part.split(None, 1)[1].strip().strip('"\'') if len(part.split(None, 1)) > 1 else ''
                if txt:
                    ops.append({'op': 'type', 'text': txt[:30]})
            elif verb in ('skip', 'noop', 'continue'):
                ops.append({'op': 'continue'})
        except (ValueError, IndexError):
            continue
        if len(ops) >= 12:
            break
    return ops


def fmt_program(ops):
    out = []
    for o in ops:
        k = o['op']
        if k == 'continue':
            out.append('continue')
        elif k == 'click':
            out.append(f"{'click' if o['b'] == 'left' else 'rclick'} {o['xy'][0]} {o['xy'][1]}")
        elif k == 'clicks':
            out.append(f"clicks {o['xy'][0]} {o['xy'][1]} {o['n']}")
        elif k == 'move':
            out.append(f"move {o['xy'][0]} {o['xy'][1]}" + (f" {o['ms']}" if o['ms'] else ''))
        elif k == 'drag':
            out.append(f"drag {o['xy'][0]} {o['xy'][1]} {o['xy2'][0]} {o['xy2'][1]}")
        elif k == 'tap':
            out.append(f"tap {o['key']}" + (f" {o['n']}" if o['n'] > 1 else ''))
        elif k == 'hold':
            ks = '+'.join(o['keys'] + (['Mouse'] if 'left' in o['buttons'] else []))
            s = f"hold {ks}"
            if o['xy']:
                s += f" {o['xy'][0]} {o['xy'][1]}"
            if o['ms'] is not None:
                s += f" {o['ms']}"
            out.append(s)
        elif k == 'wait':
            out.append(f"wait {o['ms']}")
        elif k == 'type':
            out.append(f"type {o['text']}")
    return '; '.join(out)


# ---- tool schemas (read at runtime, so compiled actions respect the harness' bounds) -----------------
def _as_dict(t):
    if isinstance(t, dict):
        return t
    if hasattr(t, 'model_dump'):
        return t.model_dump()
    if dataclasses.is_dataclass(t):
        return dataclasses.asdict(t)
    return dict(getattr(t, '__dict__', {}))


class ToolTable:
    def __init__(self, tools):
        self.schemas = {}
        for t in tools or []:
            d = _as_dict(t)
            name = d.get('name') or getattr(t, 'name', None)
            schema = d.get('parameters') or d.get('input_schema') or d.get('schema') or d.get('params') or {}
            if name:
                self.schemas[str(name)] = schema if isinstance(schema, dict) else {}
        self.has = lambda n: (not self.schemas) or n in self.schemas
        seq = self.schemas.get('sequence', {})
        steps = (seq.get('properties') or {}).get('steps') or {}
        items = steps.get('items') or {}
        self.step_fields = set((items.get('properties') or {}).keys())
        self.max_steps = int(steps.get('maxItems') or 200)
        dur = (items.get('properties') or {}).get('duration_ms') or {}
        self.max_dur = int(dur.get('maximum') or 10000)

    def props(self, name):
        return set(((self.schemas.get(name) or {}).get('properties') or {}).keys())


# ---- compile ops -> Action -----------------------------------------------------------------------------
class Compiler:
    def __init__(self, W, H, table, pointer):
        self.W, self.H, self.t = W, H, table
        self.px, self.py = pointer if pointer else (W // 2, H // 2)

    def pix(self, xy):
        return (int(round(xy[0] * (self.W - 1) / 999)), int(round(xy[1] * (self.H - 1) / 999)))

    def single(self, o):
        """One op as a plain tool, when that is enough (fewer moving parts, never refused for length)."""
        t = self.t
        if o['op'] == 'click' and t.has('click'):
            x, y = self.pix(o['xy'])
            p = {'x': x, 'y': y}
            if o['b'] != 'left':
                if 'button' not in t.props('click'):
                    return None
                p['button'] = o['b']
            self.px, self.py = x, y
            return Action('click', p)
        if o['op'] == 'tap' and o['n'] == 1 and t.has('press_key'):
            return Action('press_key', {'key': o['key']})
        if o['op'] == 'move' and not o['ms'] and t.has('move'):
            x, y = self.pix(o['xy'])
            self.px, self.py = x, y
            return Action('move', {'x': x, 'y': y})
        if o['op'] == 'drag' and t.has('drag'):
            props = t.props('drag')
            (x1, y1), (x2, y2) = self.pix(o['xy']), self.pix(o['xy2'])
            if {'x1', 'y1', 'x2', 'y2'} <= props:
                self.px, self.py = x2, y2
                return Action('drag', {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
        return None

    def sequence(self, ops):
        steps, total = [], 0
        cap = max(4, self.t.max_steps) - 1

        def add(keys, buttons, ms, pos=None):
            nonlocal total
            if len(steps) >= cap:
                return
            if pos is not None:
                self.px, self.py = pos
            ms = int(_clamp(ms, 0, self.t.max_dur))
            steps.append({'keys': list(keys), 'buttons': list(buttons), 'x': self.px, 'y': self.py,
                          'duration_ms': ms})
            total += ms

        for i, o in enumerate(ops):
            if total >= MAX_PROGRAM_MS or len(steps) >= cap:
                break
            last = i == len(ops) - 1
            k = o['op']
            if k == 'click':
                add([], [], 20, self.pix(o['xy'])); add([], [o['b']], 50); add([], [], 40)
            elif k == 'clicks':
                add([], [], 20, self.pix(o['xy']))
                for _ in range(o['n']):
                    add([], ['left'], 30); add([], [], 30)
            elif k == 'move':
                add([], [], max(40, o['ms']), self.pix(o['xy']))
            elif k == 'drag':
                (x1, y1), (x2, y2) = self.pix(o['xy']), self.pix(o['xy2'])
                add([], [], 30, (x1, y1)); add([], ['left'], 60, (x1, y1))
                for j in range(1, 7):
                    add([], ['left'], 30, (x1 + (x2 - x1) * j // 6, y1 + (y2 - y1) * j // 6))
                add([], ['left'], 60, (x2, y2)); add([], [], 30)
            elif k == 'tap':
                for _ in range(o['n']):
                    add([o['key']], [], 70); add([], [], 40)
            elif k == 'hold':
                ms = o['ms'] if o['ms'] is not None else (TRAIL_HOLD_MS if last else 300)
                add(o['keys'], o['buttons'], ms, self.pix(o['xy']) if o['xy'] else None)
            elif k == 'wait':
                add([], [], o['ms'])
            elif k == 'type':
                for ch in o['text']:
                    key = 'Space' if ch == ' ' else (ch.lower() if ch.isalpha() else ch)
                    add([key], [], 50); add([], [], 30)
        steps.append({'keys': [], 'buttons': [], 'x': self.px, 'y': self.py, 'duration_ms': 0})  # release all
        if self.t.step_fields:
            steps = [{k: v for k, v in s.items() if k in self.t.step_fields} for s in steps]
        return Action('sequence', {'steps': steps})

    def fallback(self, o):
        """Without `sequence`: the first op as the closest single tool."""
        a = self.single(o)
        if a is not None:
            return a
        t = self.t
        if o['op'] in ('click', 'clicks') and t.has('click'):
            x, y = self.pix(o['xy'])
            self.px, self.py = x, y
            return Action('click', {'x': x, 'y': y})
        if o['op'] == 'tap' and t.has('press_key'):
            return Action('press_key', {'key': o['key']})
        if o['op'] == 'hold' and o['keys']:
            props = t.props('hold_keys')
            if 'hold_keys' in t.schemas and 'keys' in props:
                p = {'keys': o['keys']}
                for name in ('duration_ms', 'ms', 'duration'):
                    if name in props:
                        p[name] = o['ms'] if o['ms'] is not None else TRAIL_HOLD_MS
                return Action('hold_keys', p)
            if t.has('press_key'):
                return Action('press_key', {'key': o['keys'][0]})
        if o['op'] == 'hold' and o['xy'] and t.has('click'):
            x, y = self.pix(o['xy'])
            return Action('click', {'x': x, 'y': y})
        return Action('skip')

    def compile(self, ops, use_sequence):
        ops = [op for op in ops if op['op'] != 'continue']
        if not ops:
            return Action('skip')
        if len(ops) == 1:
            a = self.single(ops[0])
            if a is not None:
                return a
        if use_sequence and self.t.has('sequence'):
            return self.sequence(ops)
        return self.fallback(ops[0])


# ---- perception scaffolds ------------------------------------------------------------------------------
def _font(size):
    try:
        return ImageFont.load_default(size=size)
    except Exception:
        return ImageFont.load_default()


def grid_overlay(img):
    """Faint lines every 100 units of the 0-999 action grid with edge labels. The labels sit on the model's own
    normalised scale, so they anchor its coordinates without changing the mapping."""
    im = img.convert('RGBA')
    w, h = im.size
    layer = Image.new('RGBA', im.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    fs = max(10, min(w, h) // 45)
    font = _font(fs)
    for v in range(100, 1000, 100):
        x = round(v * (w - 1) / 999)
        y = round(v * (h - 1) / 999)
        d.line([(x, 0), (x, h)], fill=(255, 0, 255, 46), width=1)
        d.line([(0, y), (w, y)], fill=(255, 0, 255, 46), width=1)
        for (tx, ty) in ((x + 2, 1), (1, y + 1)):
            d.rectangle([tx - 1, ty - 1, tx + 3 * fs * 0.62, ty + fs + 1], fill=(0, 0, 0, 120))
            d.text((tx, ty), str(v), fill=(255, 255, 255, 220), font=font)
    return Image.alpha_composite(im, layer).convert('RGB')


# ---- pixels ------------------------------------------------------------------------------------------------
def _small(img):
    return np.asarray(img.convert('L').resize((96, 54), Image.BILINEAR), dtype=np.float32) / 255.0


def _screen_key(small):
    q = (small[::6, ::6] * 7).astype(np.uint8)   # coarse: robust to small HUD/timer changes
    return hashlib.md5(q.tobytes()).hexdigest()[:12]


def _jpeg_of(item):
    if isinstance(item, (bytes, bytearray)):
        return bytes(item)
    if isinstance(item, (tuple, list)):
        for v in item:
            if isinstance(v, (bytes, bytearray)):
                return bytes(v)
    for name in ('image', 'jpeg', 'data'):
        v = getattr(item, name, None)
        if isinstance(v, (bytes, bytearray)):
            return bytes(v)
    return None


def _refusal(last):
    """Error text if the previous reply was refused, else ''."""
    if last is None:
        return ''
    d = last if isinstance(last, dict) else _as_dict(last)
    if not d:
        s = str(last)
        return s[:160] if re.search(r'refus|invalid|error', s, re.I) else ''
    ok = d.get('ok', True)
    err = d.get('error') or d.get('reason') or ''
    if ok is False or err:
        return str(err or 'refused')[:160]
    return ''


# ---- brief reader (new) ---------------------------------------------------------------------------------
ARROWS = ['ArrowUp', 'ArrowDown', 'ArrowLeft', 'ArrowRight']
WASD = ['w', 's', 'a', 'd']                     # same order as ARROWS: up, down, left, right
_BRIEF_WORDS = {'space': 'Space', 'spacebar': 'Space', 'enter': 'Enter', 'return': 'Enter', 'escape': 'Escape',
                'esc': 'Escape', 'shift': 'Shift', 'tab': 'Tab', 'ctrl': 'Control', 'control': 'Control',
                'backspace': 'Backspace'}


def read_brief(info):
    """What the brief says about the controls: movement keys, other named keys, mouse use, restart keys."""
    text = f"{info.get('controls') or ''} {info.get('description') or ''}"
    low = text.lower()
    b = {'arrows': 'arrow' in low,
         'wasd': bool(re.search(r'\bwasd\b|\bw\s*[/,]\s*a\s*[/,]\s*s\s*[/,]\s*d\b', low)),
         'mouse': bool(re.search(r'mouse|click|aim|cursor|pointer|drag', low)),
         'hold_fire': bool(re.search(r'hold (down )?(the )?(left )?(mouse )?button|fire continuously|hold to fire', low))}
    named = [k for w, k in _BRIEF_WORDS.items() if re.search(rf'\b{w}\b', low)]
    letters = set(re.findall(r"(?:press|key|keys|hit|tap|then)\s+['\"]?([A-Za-z0-9])\b(?!['\-])", text))
    letters |= set(re.findall(r"['\"]([A-Za-z0-9])['\"]", text))
    caps = set(re.findall(r"\b([A-Z0-9])(?=,\s|\s+or\s|\s+and\s|/)", text))
    caps |= set(re.findall(r"(?:,\s|\sor\s|\sand\s|/)([A-Z0-9])\b", text))
    letters |= {c for c in caps if c not in ('A', 'I')}
    restart = re.findall(r"['\"]?\b([A-Za-z])\b['\"]?\s+to\s+(?:re)?(?:start|play|try|retry)", text)
    move = ARROWS if b['arrows'] else (WASD if b['wasd'] else [])
    b['move'] = move
    b['letters'] = sorted({l.lower() for l in letters} - set(move))
    b['named'] = list(dict.fromkeys(named))
    b['start_keys'] = list(dict.fromkeys([r.lower() for r in restart] + [k for k in b['named'] if k in ('Space', 'Enter')]
                                         + ['Enter', 'Space']))[:5]
    return b


def brief_hint(b):
    """One line for the prompts."""
    parts = []
    if b['arrows'] and b['wasd']:
        parts.append('arrows or WASD to move')
    elif b['move']:
        parts.append(('arrow keys' if b['move'] is ARROWS else 'WASD') + ' to move')
    other = b['named'] + b['letters']
    if other:
        parts.append('keys ' + ' '.join(other))
    if b['mouse']:
        parts.append('mouse' + (' (hold the left button: hold Mouse X Y)' if b['hold_fire'] else ' (click)'))
    return 'KEYS from the brief: ' + ('; '.join(parts) if parts else 'not stated (try arrows, Space, Enter, mouse)')


def adapt_keys(ops, b):
    """Translate movement keys to the set this game uses (w -> ArrowUp in an arrows-only game, and back)."""
    if not b['move'] or (b['arrows'] and b['wasd']):
        return ops
    other = WASD if b['move'] is ARROWS else ARROWS
    table = {o: m for o, m in zip(other, b['move']) if o not in b['letters']}
    for o in ops:
        if o['op'] == 'tap':
            o['key'] = table.get(o['key'], o['key'])
        elif o['op'] == 'hold':
            o['keys'] = list(dict.fromkeys(table.get(k, k) for k in o['keys']))
    return ops


def find_buttons(frame, limit=5):
    """Click points (0-999 grid) for buttons or text bands on a still screen, top to bottom.

    A band is a run of rows in the middle 70% of the screen with many strong edges (letters, button borders); its
    click point is the middle of its edges. Menus usually put Play/Start first."""
    g = np.asarray(frame.convert('L').resize((200, 125), Image.BILINEAR), dtype=np.float32)
    edges = (np.abs(np.diff(g, axis=0))[:, :-1] + np.abs(np.diff(g, axis=1))[:-1, :]) > 40
    rows, cols = edges.shape
    x0, x1 = int(cols * 0.15), int(cols * 0.85)
    count = edges[:, x0:x1].sum(axis=1)
    out, start = [], None
    for r in range(rows + 1):
        hot = r < rows and count[r] >= 3
        if hot and start is None:
            start = r
        elif not hot and start is not None:
            if r - start >= 2:
                ys, xs = np.nonzero(edges[start:r, x0:x1])
                out.append((int((xs.mean() + x0) * 999 / (cols - 1)), int((start + r) / 2 * 999 / (rows - 1))))
            start = None
    return out[:limit]


# deterministic probes for a screen that ignores the model's inputs (menus, focus, game-over)
UNSTICK = [
    'click 500 500', 'tap Enter', 'tap Space', 'click 500 620', 'click 500 760', 'tap r',
    'click 500 380', 'hold ArrowRight 600; tap ArrowUp', 'click 500 880', 'tap Escape',
]


class Agent:
    def __init__(self):
        self.last_elapsed = float('inf')
        self.table, self._tools_id = ToolTable([]), None
        self._new_game()

    # -- per-game state --------------------------------------------------------------------------------
    def _new_game(self):
        self.steps = 0
        self.notes = ''
        self.plan_t = None
        self.log = deque(maxlen=8)          # {'t', 'cmd', 'change', 'key'}
        self.frames = deque(maxlen=48)      # (elapsed, PIL image)
        self.prev_small = None
        self.frozen = 0
        self.scores = deque(maxlen=6)
        self.pointer = None
        self.use_sequence = True
        self.seq_refusals = 0
        self.last_tool = None
        self.noop = {}                      # screen_key -> set of commands that changed nothing there
        self.unstick_i = 0
        self.last_action = 'none'
        self.mode = ''
        self.brief = None                   # read_brief(info), on the first observation of the game
        self.buttons = {}                   # screen_key -> click points found on that screen
        self.last_good = None               # the last program after which the screen moved

    # -- public interface ------------------------------------------------------------------------------
    def step(self, obs, tools):
        try:
            action = self._step(obs, tools)
        except Exception as e:
            print('agent error:', repr(e)[:300])
            action = Action('skip')
        return action

    # -- core --------------------------------------------------------------------------------------------
    def _step(self, obs, tools):
        el = float(getattr(obs, 'elapsed_s', 0.0) or 0.0)
        if el < self.last_elapsed - 0.5:
            self._new_game()
        self.last_elapsed = el
        self.input_feedback = describe_inputs(obs)
        if id(tools) != self._tools_id:
            self.table, self._tools_id = ToolTable(tools), id(tools)

        info = getattr(obs, 'info', None) or {}
        if self.brief is None:
            self.brief = read_brief(info)
        frame = Image.open(io.BytesIO(obs.image)).convert('RGB')
        W = int(info.get('width') or frame.size[0])
        H = int(info.get('height') or frame.size[1])

        # screen-change feedback for the previous input
        small = _small(frame)
        delta = 1.0 if self.prev_small is None else float(np.abs(small - self.prev_small).mean())
        motion = 0.0
        recent = list(getattr(obs, 'frames', None) or [])
        if len(recent) >= 2:
            a, b = _jpeg_of(recent[0]), _jpeg_of(recent[-1])
            if a and b:
                motion = float(np.abs(_small(Image.open(io.BytesIO(b))) - _small(Image.open(io.BytesIO(a)))).mean())
        active = max(delta, motion) if self.prev_small is not None else 1.0
        if self.log and self.log[-1]['change'] is None:
            self.log[-1]['change'] = delta
            if active >= FROZEN_EPS and self.log[-1].get('ops'):
                self.last_good = self.log[-1]['ops']
            if active < FROZEN_EPS and self.log[-1]['key']:
                self.noop.setdefault(self.log[-1]['key'], set()).add(self.log[-1]['cmd'])
        self.prev_small = small
        self.frozen = self.frozen + 1 if active < FROZEN_EPS else 0
        skey = _screen_key(small)

        refused = _refusal(getattr(obs, 'last_action', None))
        if refused and self.last_tool == 'sequence':
            self.seq_refusals += 1
            if self.seq_refusals >= 3:
                self.use_sequence = False      # harness keeps refusing programs: degrade to single tools
        elif self.last_tool == 'sequence':
            self.seq_refusals = 0

        img = frame.copy()
        img.thumbnail((IMG_SIDE, IMG_SIDE))
        if GRID_OVERLAY:
            img = grid_overlay(img)
        self.frames.append((el, img))
        self.steps += 1
        self._frame, self._skey = frame, skey

        if not MANAGER.ready():   # model server still loading or restarting: cheap probes, no model
            self.mode = 'no-model'
            ops = parse_program(self._unstick_text())
            action = Compiler(W, H, self.table, self.pointer).compile(ops, self.use_sequence)
            self.last_tool, self.last_action = action.tool, f'[no-model] {fmt_program(ops)}'
            self.log.append({'t': el, 'cmd': fmt_program(ops), 'change': None, 'key': None})
            return action

        since = None if self.plan_t is None else el - self.plan_t
        need_plan = (since is None or since >= T('PLAN_EVERY_S')
                     or (self.frozen >= 3 and since >= T('PLAN_MIN_GAP_S'))
                     or (delta > BIG_CHANGE and since >= T('PLAN_MIN_GAP_S'))
                     or (self._looping() and since >= T('PLAN_MIN_GAP_S')))

        if self.frozen >= UNSTICK_AFTER and self.frozen % 2 == 0 and not need_plan:
            self.mode = 'unstick'
            text = self._unstick_text()
        elif need_plan:
            self.mode = 'plan'
            text = self._plan(obs, info, el, refused)
            self.plan_t = el
        else:
            self.mode = 'act'
            text = MANAGER.generate(SYSTEM_ACT, [img], self._act_prompt(info, el, refused, skey), '',
                                    T('ACT_MAX_TOKENS'), 'line')

        ops = adapt_keys(parse_program(text), self.brief)
        if not ops:                          # empty / unparseable reply: keep playing instead of skip
            if self.last_good and self.frozen < 2:
                ops, self.mode = copy.deepcopy(self.last_good), self.mode + '+repeat'
            else:
                ops, self.mode = parse_program(self._unstick_text()), self.mode + '+probe'
        comp = Compiler(W, H, self.table, self.pointer)
        action = comp.compile(ops, self.use_sequence)
        self.pointer = (comp.px, comp.py)
        self.last_tool = action.tool
        cmd = fmt_program(ops) or 'skip'
        self.last_action = f'[{self.mode}] {cmd}'
        self.log.append({'t': el, 'cmd': cmd, 'change': None, 'key': skey if ops else None,
                         'ops': copy.deepcopy(ops) if self.mode in ('act', 'plan') else None})
        return action

    def _unstick_text(self):
        """The next scripted probe: buttons found on this screen, the brief's start keys, then the generic list."""
        skey = self._skey
        if skey not in self.buttons:
            try:
                self.buttons[skey] = find_buttons(self._frame)
            except Exception:
                self.buttons[skey] = []
        cands = [f'click {x} {y}' for x, y in self.buttons[skey]]
        cands += [f'tap {k}' for k in (self.brief or {}).get('start_keys', [])]
        cands += [u for u in UNSTICK if u not in cands]
        text = cands[self.unstick_i % len(cands)]
        self.unstick_i += 1
        return text

    def _looping(self):
        if len(self.log) < 4:
            return False
        last = list(self.log)[-4:]
        same = len({e['cmd'] for e in last}) == 1
        changes = [e['change'] for e in last if e['change'] is not None]
        return same and bool(changes) and max(changes) < 0.02

    # -- prompts ---------------------------------------------------------------------------------------
    def _brief(self, info):
        parts = []
        for key in ('title', 'genre'):
            if info.get(key):
                parts.append(f'{key.capitalize()}: {info[key]}')
        parts.append(f"Brief: {info.get('description', '')}")
        parts.append(f"Controls: {info.get('controls', '')}")
        if self.brief:
            parts.append(brief_hint(self.brief))
        return '\n'.join(parts)

    def _log_text(self, n):
        rows = []
        for e in list(self.log)[-n:]:
            ch = '?' if e['change'] is None else f"{100 * e['change']:.1f}%"
            rows.append(f"  t={e['t']:.0f}s: {e['cmd'][:90]} -> screen change {ch}")
        return '\n'.join(rows) if rows else '  (none yet)'

    def _warnings(self, refused, skey):
        w = []
        if refused:
            w.append(f'The game refused your previous input: {refused}')
        if self.frozen >= 3:
            w.append(f'The screen has not changed for {self.frozen} replies: your inputs are not working. '
                     'Try something different (click a visible button, Enter, Space, other keys).')
        if self._looping():
            w.append('You repeated the same input 4 times with almost no effect. Change it.')
        dead = sorted(self.noop.get(skey, ()))[:5]
        if dead:
            w.append('On this screen these inputs did nothing: ' + ' | '.join(d[:60] for d in dead))
        return '\n'.join(w)

    def _time(self, info, el):
        rem = info.get('remaining_s')
        return f'Game time: {el:.0f} s played' + (f', {float(rem):.0f} s left.' if rem is not None else '.') + ('\n' + self.input_feedback if self.input_feedback else '')

    def _act_prompt(self, info, el, refused, skey):
        text = [self._brief(info), self._time(info, el)]
        if self.notes:
            text.append('NOTES from your coach:\n' + self.notes)
        text.append('Recent inputs (oldest first):\n' + self._log_text(4))
        w = self._warnings(refused, skey)
        if w:
            text.append('WARNING: ' + w)
        text.append('The screenshot is NOW. Your commands:')
        return '\n'.join(text)

    def _plan(self, obs, info, el, refused):
        imgs = self._plan_images(el)
        text = [self._brief(info), self._time(info, el)]
        if self.notes:
            text.append('Your previous notes:\n' + self.notes)
        if self.scores:
            text.append('Score readings so far: ' + ', '.join(f'{t:.0f}s: {s}' for t, s in self.scores))
        text.append('Recent inputs (oldest first):\n' + self._log_text(8))
        w = self._warnings(refused, _screen_key(self.prev_small))
        if w:
            text.append('WARNING: ' + w)
        ages = ', '.join(f'{el - t:.0f} s ago' for t, _ in imgs[:-1])
        text.append(f'Screenshots: {ages + ", " if ages else ""}and NOW (last).')
        out = MANAGER.generate(SYSTEM_PLAN, [im for _, im in imgs], '\n'.join(text), 'SCREEN:',
                               T('PLAN_MAX_TOKENS'), 'plan')

        fields = {}
        for line in out.splitlines():
            m = re.match(r'\s*(SCREEN|SCORE|LEARNED|PLAN|ACT)\s*:\s*(.*)', line)
            if m:
                fields[m.group(1)] = m.group(2).strip()
        notes = [f'{k}: {fields[k][:220]}' for k in ('SCREEN', 'LEARNED', 'PLAN') if fields.get(k)]
        if notes:
            self.notes = '\n'.join(notes)
        sc = fields.get('SCORE', '').strip()
        if sc and sc != '?':
            self.scores.append((el, sc[:24]))
        return 'ACT: ' + fields.get('ACT', '')

    def _plan_images(self, el):
        frames = list(self.frames)
        now = frames[-1]
        chosen = []
        for age in (8.0, 3.0):
            older = [f for f in frames[:-1] if f[0] <= el - age + 0.6]
            if older and all(older[-1][0] != c[0] for c in chosen):
                chosen.append(older[-1])
        return chosen + [now]

### Steering on Off Line

Estimate obstacle speed between frames and compare three successive steering inputs, including the delay before the first input can take effect.

In [ ]:
import io
import numpy as np
from PIL import Image
from scipy import ndimage
from gga.protocol import Action


class LineDriver:
    """Steer using the line, player and falling objects visible in Off Line."""

    def __init__(self):
        self.player = (128, 232)
        self.last_delta = 0
        self.previous_objects = []
        self.previous_time = None
        self.fall_speed = 2.
        self.moves = np.array([-24, -12, 0, 12, 24])
        grids = np.meshgrid(self.moves, self.moves, self.moves, indexing='ij')
        self.programs = np.stack(grids, axis=-1).reshape(-1, 3)
        ticks = np.arange(1, 13)
        starts = self.programs.cumsum(axis=1) - self.programs
        offsets = np.sign(self.programs[:,:,None]) * np.minimum(2*ticks, np.abs(self.programs[:,:,None]))
        self.trajectories = (starts[:,:,None] + offsets).reshape(-1, 36)

    @staticmethod
    def regions(mask):
        labels, _ = ndimage.label(mask)
        objects = ndimage.find_objects(labels)
        return [(x.start, x.stop, y.start, y.stop) for obj in objects if obj
                for y, x in [obj]]

    def step(self, obs, tools):
        with Image.open(io.BytesIO(obs.image)) as frame:
            # The public game centers a square canvas inside a white viewport.
            shrink = np.asarray(frame.convert('RGB').resize((320, 320)), dtype=np.int16)
        mid = shrink[160]
        field = np.flatnonzero((mid[:, 0] - mid[:, 2] > 25) | (mid[:, 2] - mid[:, 0] > 25))
        if field.size < 80:
            return Action('press_key', {'key':'Space'})
        left, right = field[0], field[-1] + 1
        top = (320 - (right-left))//2
        tile = shrink[top:top+right-left, left:right]
        r, g, b = tile.transpose(2, 0, 1)
        purple = (b > r+30) & (r > g+9)
        green = (g > r+20) & (g > b+70)
        line = (purple | green).sum(axis=0)
        if line.max() < len(tile)*.46:
            self.player = (128, 232)
            self.last_delta = 0
            self.previous_objects = []
            self.previous_time = None
            self.fall_speed = 2.
            return Action('press_key', {'key':'Space', 'hold_ms':50})
        line_x = int(np.argmax(line))
        blue = (b > r+70) & (b > g+25)
        black = (r < 50) & (g < 50) & (b < 55)
        player_mask = blue | purple | black
        player_mask[:, line > len(tile)*.46] &= ~purple[:, line > len(tile)*.46]
        candidates = [(x0, x1, y0, y1) for x0, x1, y0, y1 in self.regions(player_mask)
                      if 12 <= x1-x0 <= 25 and 10 <= y1-y0 <= 26 and y0 > 25]
        if candidates:
            px0, py0 = self.player
            x0, x1, y0, y1 = min(candidates, key=lambda p: abs((p[0]+p[1])/2-px0)+abs((p[2]+p[3])/2-py0))
            self.player = ((x0+x1-1)/2, (y0+y1-1)/2)
        px, py = self.player
        orange = (r > 195) & (g > 88) & (g < 199) & (b < 72)
        objects = [(x0, x1, y0, y1) for x0,x1,y0,y1 in self.regions(orange)
                   if x1-x0 > 5 and y1-y0 > 5]
        elapsed = obs.elapsed_s - self.previous_time if self.previous_time is not None else 0
        if 0 < elapsed < .8:
            ticks_elapsed = 60 * elapsed
            velocities = [(y0 - old_y0) / ticks_elapsed for x0,x1,y0,y1 in objects
                          for old_x0,old_x1,old_y0,old_y1 in self.previous_objects
                          if abs(x0-old_x0) <= 1 and abs(x1-old_x1) <= 1
                          and abs((y1-y0)-(old_y1-old_y0)) <= 2
                          and 1.2 < (y0-old_y0)/ticks_elapsed < 8]
            if velocities:
                self.fall_speed = .65*self.fall_speed + .35*float(np.median(velocities))
        self.previous_objects, self.previous_time = objects, obs.elapsed_s
        threats = [(x0,x1,y0,y1) for x0,x1,y0,y1 in objects
                   if y0 < py+12 and y1 > py-48*self.fall_speed-12]
        # The observation precedes the drawing of the last input. The proposed
        # input can start only at the next grid point, 12 game ticks later.
        ticks = np.arange(13, 49)
        path = px + self.last_delta + self.trajectories
        cost = .32*(np.abs(path-(line_x+1)) > 8)
        for x0, x1, y0, y1 in threats:
            near_y = (y0 + self.fall_speed*ticks < py + 8) & (y1 + self.fall_speed*ticks > py - 8)
            cost += ((path > x0-8) & (path < x1+8) & near_y[None,:]).astype(float)
        rating = cost.sum(axis=1) + .04*np.abs(path[:,-1]-line_x) + .003*np.abs(self.programs).sum(axis=1)
        delta = int(self.programs[np.argmin(rating),0])
        self.last_delta = delta
        if not delta:
            return Action('skip')
        return Action('press_key', {'key':'ArrowRight' if delta > 0 else 'ArrowLeft', 'hold_ms':round(abs(delta)*1000/120)})

### Flying in Asteroid Capitalist

Match sprites between frames to estimate their motion. At each decision the controller compares three short flight paths and keeps firing while the game advances.

In [ ]:
import io

import numpy as np
from PIL import Image
from scipy import ndimage
from gga.protocol import Action


class ArcadeTracker:
    """Track moving sprites and choose a short flight path from the current screen."""

    def __init__(self, sweep=False):
        self.sweep = sweep
        self.player = np.array([200., 112.])
        self.previous = {}
        self.last_time = None
        self.direction = 1

    @staticmethod
    def centers(mask, minimum, radius):
        labels, _ = ndimage.label(ndimage.maximum_filter(mask, size=radius))
        points = []
        for i, box in enumerate(ndimage.find_objects(labels), 1):
            if box is None:
                continue
            ys, xs = box
            if np.count_nonzero(mask[box] & (labels[box] == i)) >= minimum:
                points.append(((xs.start+xs.stop-1)/2, (ys.start+ys.stop-1)/2))
        return np.asarray(points, dtype=float).reshape(-1, 2)

    def tracks(self, group, positions, dt, expected_velocity, gate):
        previous = self.previous.get(group)
        velocities = np.tile(expected_velocity, (len(positions), 1)).astype(float)
        if previous is not None and len(previous[0]) and len(positions) and .05 <= dt <= .6:
            old_pos, old_speed = previous
            extrapolated = old_pos + old_speed*dt
            distances = np.linalg.norm(positions[:, None, :] - extrapolated[None, :, :], axis=2)
            while distances.size:
                i, j = np.unravel_index(np.argmin(distances), distances.shape)
                if distances[i, j] > gate:
                    break
                velocities[i] = .65*(positions[i]-old_pos[j])/dt + .35*old_speed[j]
                distances[i, :] = np.inf
                distances[:, j] = np.inf
        self.previous[group] = positions, velocities
        return velocities

    def step(self, obs, tools):
        with Image.open(io.BytesIO(obs.image)) as screen:
            w, h = screen.size
            view = np.asarray(screen.convert('RGB').resize((400, round(h*400/w))), dtype=np.int16)
        r, g, b = view.transpose(2, 0, 1)
        white = (r > 175) & (g > 175) & (b > 175)
        if white[8:32, 60:340].sum() > 230:
            self.previous.clear()
            self.last_time = None
            return Action('click', {'x': w//2, 'y': round(w*.39)})
        white_near = ndimage.maximum_filter(white, size=5)
        green = (g > r+24) & (g > b+32) & (g > 105) & white_near
        green[:10] = False
        ships = self.centers(green, 3, 5)
        if len(ships):
            self.player = ships[np.argmin(np.linalg.norm(ships-self.player, axis=1))]
        red = (r > 100) & (r > 1.5*g) & (r > 1.35*b) & white_near
        red[:8] = False
        rivals = self.centers(red, 4, 5)
        blue = (b > 130) & (b > r+55) & (b > g+12)
        blue[:8] = False
        bolts = self.centers(blue, 2, 3)
        dt = obs.elapsed_s-self.last_time if self.last_time is not None else 0.
        self.last_time = obs.elapsed_s
        rival_v = self.tracks('ships', rivals, dt, (-37.5, 0.), 17.)
        bolt_v = self.tracks('bolts', bolts, dt, (-165., 0.), 35.)
        x, y = self.player
        aim = np.array([395., y])
        if len(rivals):
            # Project the next sprite position in the screenshot. Updates arrive on a 200-ms grid.
            duration = .2 + np.linalg.norm(rivals-self.player, axis=1)/185.
            predicted = rivals + rival_v*duration[:, None]
            duration = .2 + np.linalg.norm(predicted-self.player, axis=1)/185.
            predicted = rivals + rival_v*duration[:, None]
            distance = np.linalg.norm(rivals-self.player, axis=1)
            priority = distance + 65*(rivals[:, 0] < x-12) + 90*(predicted[:, 0] < 3)
            aim = predicted[np.argmin(priority)]
        if y < 22:
            self.direction = 1
        elif y > 200:
            self.direction = -1
        velocities = np.array([[0, -83.], [0, 0], [0, 83.]])
        times = np.arange(.2, 1.41, .2)
        paths = self.player + velocities[:, None, :]*times[None, :, None]
        paths[:, :, 1] = np.clip(paths[:, :, 1], 11., 213.)
        cost = np.zeros(3)
        for objects, speed, radius, weight in ((rivals, rival_v, 29., 3.), (bolts, bolt_v, 20., 6.)):
            if len(objects):
                future = objects[:, None, :] + speed[:, None, :]*times[None, :, None]
                clearance = np.linalg.norm(paths[:, None, :, :]-future[None, :, :, :], axis=-1)
                cost += weight*np.maximum(0., 1.-clearance/radius).max(axis=1).sum(axis=1)
        desired_y = (206. if self.direction > 0 else 18.) if self.sweep else 112.
        cost += np.abs(paths[:, 2, 1] - desired_y)*(.008 if self.sweep else .0015)
        keys = ['a'] if x > 48 else (['d'] if x < 20 else [])
        vertical = ('w', None, 's')[int(np.argmin(cost))]
        if vertical:
            keys.append(vertical)
        target = np.clip(np.rint(aim*w/400), [0, 0], [w-1, h-1]).astype(int)
        return Action('sequence', {'steps': [dict(keys=keys, buttons=['left'], x=int(target[0]),
                                                  y=int(target[1]), duration_ms=450)]})

### Excavation and defense in Cavebots

Read the visible mineral deposits and bot caches. Order digs and try construction shortcuts at sites uncovered near the core.

In [ ]:
import io

import numpy as np
from PIL import Image
from scipy import ndimage
from gga.protocol import Action


class CaveBuilder:
    """Excavate visible bot caches and building sites near the starting tunnel."""
    def __init__(self):
        self.tried = {}
        self.sites = []
        self.next_order = 0
        self.orders = 0

    def step(self, obs, tools):
        with Image.open(io.BytesIO(obs.image)) as frame:
            w, h = frame.size
            view = np.asarray(frame.convert('RGB').resize((400, 225)), dtype=np.int16)
        r, g, b = view.transpose(2,0,1)
        if (view[:15].max(axis=2) < 18).mean() > .90:
            if (view[70:98, 114:286].mean(axis=2) > 28).mean() > .6:
                self.__init__()
                return Action('click', {'x':w//2, 'y':round(h*.37)})
            return Action('click', {'x':w//2,'y':round(h*.66)})
        if obs.elapsed_s < self.next_order:
            return Action('skip')
        self.next_order = obs.elapsed_s + 3
        self.orders += 1
        # Green blocks contain bots, pale deposits yield iron, and purple rock uncovers building sites.
        green = (g > r+8) & (g > b+8) & (r > 56) & (r < 150)
        pale = (b > r+6) & (r > 127) & (g > 115) & (g < 190)
        ore = (b > r+42) & (r > 47) & (r > g+9)
        weak = (abs(r-84) < 10) & (abs(g-88) < 10) & (abs(b-111) < 12)
        stones = (r > g+25) & (r > b+20) & (r < 155) & (g > 60)
        for mask in (green, pale, ore, weak, stones):
            mask[:62] = False
            mask[195:] = False
            mask[:, 260:] = False
            mask[:, :8] = False
        if self.sites and self.orders % 4 == 0:
            x,y = self.sites[(self.orders//4-1) % len(self.sites)]
            key = ('q', 'w', 'w', 'e')[(self.orders//4-1)//len(self.sites) % 4]
            return Action('sequence', {'steps':[{'x':x, 'y':y, 'duration_ms':100},
                                                {'x':x, 'y':y, 'keys':[key], 'duration_ms':120}]})
        prospects = []
        for priority, mask in enumerate((green, ore, weak, pale, stones)):
            smooth = ndimage.binary_closing(mask, structure=np.ones((3,3)))
            labels,_ = ndimage.label(smooth)
            for box in ndimage.find_objects(labels):
                if box is None:
                    continue
                ys,xs = box
                if not 6 <= xs.stop-xs.start <= 33 or not 6 <= ys.stop-ys.start <= 31:
                    continue
                x,y = (xs.start+xs.stop)//2,(ys.start+ys.stop)//2
                if mask[ys,xs].mean() < .34:
                    continue
                pos = round(x*w/400), round(y*h/225)
                recent = min(60, min((obs.elapsed_s-t for (ox,oy),t in self.tried.items()
                                      if abs(pos[0]-ox)<30 and abs(pos[1]-oy)<30), default=100))
                weight = priority*5 + max(0, 18-recent)*4 + abs(y-119)*.2 + abs(x-159)*.08
                prospects.append((weight, priority, pos))
        if prospects:
            _, priority, pos = min(prospects)
            self.tried[pos] = obs.elapsed_s
            if priority == 1 and all(abs(pos[0]-x)+abs(pos[1]-y)>28 for x,y in self.sites):
                self.sites.append(pos)
            return Action('click', {'x':pos[0], 'y':pos[1]})
        return Action('skip')

### Battery and nutrients in Blarglike

A moving flower and the player both change square color on each move. The controller tries up to five routes over successive lives, varying the value of food and batteries against the detour to the stairs. It reads the visible score to choose a route for later lives.

In [ ]:
import io
from collections import deque

import numpy as np
from PIL import Image
from gga.protocol import Action


class DungeonExplorer:
    """Track tile colors and confirm each movement before using another battery charge."""

    moves = ((0, 1, 'ArrowRight'), (1, 0, 'ArrowDown'), (0, -1, 'ArrowLeft'), (-1, 0, 'ArrowUp'))

    def __init__(self, harvest=False, parity=False, flip=False, route=False,
                 flower_bonus=8., battery_bonus=0., detour=.8, flip_radius=4):
        self.harvest = harvest
        self.parity = parity
        self.flip = flip
        self.route = route
        self.flower_bonus = flower_bonus
        self.battery_bonus = battery_bonus
        self.detour = detour
        self.flip_radius = flip_radius
        self.last_bump = -10.
        self.origin = None
        self.sent_at = -1.
        self.blocked = {}
        self.last_board = None
        self.last_start = -10.
        self.visits = {}

    def board(self, jpeg):
        with Image.open(io.BytesIO(jpeg)) as image:
            w, h = image.size
            hud = np.asarray(image.crop((round(173*w/1600), round(9*h/1000),
                                        round(243*w/1600), round(26*h/1000))).convert('L')
                             .resize((70, 17))) > 125
            single_digit = not hud[:, 10:20].any()
            first_digit, zero_digit = hud[:, :10], hud[:, 50:60]
            low_battery = single_digit and first_digit.any()
            flat_battery = low_battery and np.count_nonzero(first_digit ^ zero_digit) < 7
            # Detect the visible playfield inside the black margins.
            small = np.asarray(image.resize((320, 200), Image.Resampling.NEAREST), dtype=np.int16)
            r, g, b = small.transpose(2, 0, 1)
            field = (b > r + 14) & (g > 12) & (b > 43)
            rows = np.flatnonzero(field.mean(axis=1) > .48)
            cols = np.flatnonzero(field.mean(axis=0) > .45)
            if rows.size < 110 or cols.size < 210:
                return None
            bounds = (round(cols[0]*w/320), round(rows[0]*h/200),
                      round((cols[-1]+1)*w/320), round((rows[-1]+1)*h/200))
            grid = np.asarray(image.crop(bounds).resize((480, 320)), dtype=np.int16)
        r, g, b = grid.transpose(2, 0, 1)

        def counts(red, green, blue, tolerance):
            mask = (abs(r-red) < tolerance) & (abs(g-green) < tolerance) & (abs(b-blue) < tolerance)
            return mask.reshape(20, 16, 30, 16).sum(axis=(1, 3))

        walls = counts(104, 141, 178, 24) >= 30
        walls[[0, -1], :] = True
        walls[:, [0, -1]] = True
        yellow = counts(254, 231, 97, 45)
        if yellow.max() < 8:
            return None
        player = tuple(map(int, np.unravel_index(yellow.argmax(), yellow.shape)))
        pink_pixels = (r > 150) & (r > g+54) & (b > g-23) & (b < g+29) & (b > 65)
        pink = pink_pixels.reshape(20, 16, 30, 16).sum(axis=(1, 3)) > 7
        orange = counts(247, 118, 34, 40) > 9
        stair_pixels = (r > 135) & (g > r+8) & (b > g+5)
        blue = stair_pixels.reshape(20, 16, 30, 16).sum(axis=(1, 3))
        candidates = (counts(160, 194, 229, 20) >= 7) & (blue >= 21)
        blue = np.where(candidates, blue, 0)
        exit_tile = tuple(map(int, np.unravel_index(blue.argmax(), blue.shape))) if blue.any() else None
        walls[pink | orange] = False
        walls[player] = False
        if exit_tile is not None:
            walls[exit_tile] = False
        return player, walls, pink, orange, exit_tile, low_battery, flat_battery

    def paths(self, start, walls, end, now):
        queue = deque([start])
        paths = {start: (0, None, None)}
        while queue:
            cell = queue.popleft()
            distance, first_key, first_cell = paths[cell]
            if cell == end and cell != start:
                continue
            for dy, dx, key in self.moves:
                other = cell[0] + dy, cell[1] + dx
                if not (0 <= other[0] < 20 and 0 <= other[1] < 30):
                    continue
                if walls[other] or other in paths or self.blocked.get((cell, other), -1) > now:
                    continue
                paths[other] = distance+1, first_key or key, first_cell or other
                queue.append(other)
        return paths

    def step(self, obs, tools):
        now = obs.elapsed_s
        state = self.board(obs.image)
        if state is None or self.harvest and state[-1]:
            self.origin = self.last_board = None
            if now - self.last_start >= .8:
                self.last_start = now
                return Action('press_key', {'key': 'r', 'hold_ms': 50})
            return Action('skip')
        start, walls, flowers, batteries, exit_tile, low_battery, _ = state
        if self.flip and now - self.last_bump < .38:
            return Action('skip')
        # A floor transition invalidates any previously rejected wall crossing.
        layout = walls[2:-2:3, 2:-2:3].tobytes()
        if self.last_board != layout:
            self.blocked.clear()
            self.visits.clear()
            self.last_board = layout
        if self.origin == start:
            if now-self.sent_at < .57:
                return Action('skip')
            self.blocked[start, self.expected] = now + 3.
        self.origin = None
        paths = self.paths(start, walls, exit_tile, now)
        if self.flip and now-self.last_bump > 2.0:
            nearby = [paths[cell][0] for cell in zip(*np.nonzero(flowers)) if cell != start and cell in paths]
            even = min((d for d in nearby if d % 2 == 0), default=100)
            odd = min((d for d in nearby if d % 2 == 1), default=100)
            # Spending one charge on a wall lets mobile targets change square color
            # while the player stays put. A flower restores ten charges on contact.
            if even <= self.flip_radius and odd > even+1 and (exit_tile not in paths or paths[exit_tile][0] > 1):
                for dy, dx, key in self.moves:
                    y, x = start[0]+dy, start[1]+dx
                    if 0 <= y < 20 and 0 <= x < 30 and walls[y, x]:
                        self.last_bump = now
                        return Action('press_key', {'key': key, 'hold_ms': 50})
        targets = []
        through_exit = self.paths(exit_tile, walls, None, now) if self.route and exit_tile in paths else {}
        if exit_tile in paths and exit_tile != start:
            distance = paths[exit_tile][0]
            targets.append((distance+2.0 if self.harvest else distance*.70, exit_tile))
        bonuses = ((flowers, self.flower_bonus), (batteries, self.battery_bonus+(5.5 if low_battery else 3.5))) if self.harvest else ((flowers, 3.5), (batteries, 1.5))
        for mask, bonus in bonuses:
            for cell in zip(*np.nonzero(mask)):
                if cell in paths and cell != start:
                    distance = paths[cell][0]
                    # A moving flower normally changes square color every turn too.
                    # Only an odd-distance flower can be adjacent before a later step.
                    if self.parity and mask is flowers and distance % 2 == 0:
                        continue
                    rank = distance - bonus
                    if through_exit and cell in through_exit:
                        extra = max(0, distance + through_exit[cell][0] - paths[exit_tile][0])
                        rank += self.detour * extra
                    targets.append((rank, cell))
        if not targets:
            for cell, (distance, _, _) in paths.items():
                if distance:
                    targets.append((distance+self.visits.get(cell, 0)*4, cell))
        if not targets:
            return Action('press_key', {'key': 'r', 'hold_ms': 50})
        _, target = min(targets)
        _, key, expected = paths[target]
        self.visits[start] = self.visits.get(start, 0) + 1
        self.origin, self.expected, self.sent_at = start, expected, now
        return Action('press_key', {'key': key, 'hold_ms': 50})


import base64
import io

import numpy as np
from PIL import Image


class DungeonPortfolio:
    """Compare routes across lives using the battery and score shown on screen."""

    glyphs = np.unpackbits(np.frombuffer(base64.b64decode(
        'eOaHh4GBgYGHh+Z4ePgYGBgYGBgYGBj/PEIDAwMCBhw4OGD/PkIDAwIcBgMBAYN8'
        'Bh4mJiZGRv8GBgYffkBAQEB+QwEBAQP8H3Dg4ID+54GBged+/wEGBgYOCAgYGBAQ'
        'foeBgYd4h4GBgYd+eIaHh4eHeQcGBg7w'), dtype=np.uint8)).reshape(10, 12, 8).astype(bool)

    def __init__(self, policy_factory, explore_routes=False):
        self.policy_factory = policy_factory
        self.routes = [{}, dict(flip=True, route=True)]
        if explore_routes:
            self.routes.extend([
                dict(flip=True, route=True, flower_bonus=12., battery_bonus=4., detour=.35, flip_radius=6),
                dict(flip=True, route=True, flower_bonus=5., battery_bonus=1., detour=1.3),
                dict(flip=True, flower_bonus=10., battery_bonus=5., flip_radius=6),
            ])
        self.best = [0]*len(self.routes)
        self.lives = 0
        self.choice = 0
        self.policy = self.new_policy()
        self.last_read = -1.
        self.last_health = None
        self.last_score = 0
        self.dead = False

    def new_policy(self):
        return self.policy_factory(harvest=True, parity=True, **self.routes[self.choice])

    @classmethod
    def counters(cls, frame):
        w, h = frame.size
        header = frame.crop((0, round(5*h/1000), round(900*w/1600), round(32*h/1000)))
        mask = np.asarray(header.convert('L').resize((900, 27))) > 140
        bars = np.flatnonzero(mask.sum(axis=0) >= 14)
        if len(bars) != 4:
            return None
        hp_length = round((bars[0]-237)/10)
        score_length = round((bars[3]-bars[2])/10) - 10
        if hp_length not in (1, 2) or not 1 <= score_length <= 6:
            return None

        def number(left, length):
            value = 0
            for k in range(length):
                sample = mask[:, left+10*k:left+10*k+10]
                yy, xx = np.nonzero(sample)
                if not xx.size:
                    return None
                glyph = Image.fromarray(sample[yy.min():yy.max()+1, xx.min():xx.max()+1])
                errors = np.mean(cls.glyphs != np.asarray(glyph.resize((8, 12), Image.Resampling.NEAREST)), axis=(1, 2))
                digit = int(errors.argmin())
                if errors[digit] > .16:
                    return None
                value = 10*value + digit
            return value

        hp, score = number(173, hp_length), number(int(bars[2])+86, score_length)
        return (hp, score) if hp is not None and hp <= 50 and score is not None else None

    def observe(self, health, score):
        restarted = (self.dead or score < self.last_score or
                     self.last_health is not None and self.last_health <= 3 and health == 50)
        if health > 0 and restarted:
            self.lives += 1
            self.choice = self.lives if self.lives < len(self.routes) else int(np.argmax(self.best))
            self.policy = self.new_policy()
        self.best[self.choice] = max(self.best[self.choice], score)
        self.last_health, self.last_score, self.dead = health, score, health == 0

    def step(self, obs, tools):
        if obs.elapsed_s-self.last_read >= .35:
            self.last_read = obs.elapsed_s
            with Image.open(io.BytesIO(obs.image)) as frame:
                values = self.counters(frame)
            if values:
                self.observe(*values)
        return self.policy.step(obs, tools)

In [ ]:
BriefPlanner = Agent


class Agent:
    def __init__(self):
        self.general = None
        self.specialist = None
        self.game_key = None
        self.elapsed = -1

    def step(self, obs, tools):
        info = obs.info
        game_id = info.get('game_id')
        key = game_id, info.get('description'), info.get('controls')
        if self.game_key != key or obs.elapsed_s < self.elapsed:
            self.game_key = key
            self.general = BriefPlanner()
            self.specialist = LineDriver() if game_id == 'off_line' else None
            if game_id == 'asteroid_capitalist':
                self.specialist = ArcadeTracker()
            if game_id == 'cavebots':
                self.specialist = CaveBuilder()
            if game_id == 'blarglike':
                self.specialist = DungeonPortfolio(DungeonExplorer, explore_routes=True)
        self.elapsed = obs.elapsed_s
        if self.specialist is not None:
            return self.specialist.step(obs, tools)
        return self.general.step(obs, tools)

### Unknown-game diagnostic

A generated menu tests the planner and actor through the competition action interface.

In [ ]:
if not rerun:
    import json
    from gga.protocol import Observation, tools as game_tools, validate
    scene = Image.new('RGB', (480, 320), '#14232b')
    paint = ImageDraw.Draw(scene)
    paint.rectangle((100, 140, 270, 210), fill='#28794e')
    paint.text((158, 163), 'PLAY', fill='white', font_size=24)
    image = io.BytesIO()
    scene.save(image, format='JPEG')
    hidden = dict(width=480, height=320, controls='Click Play to begin. Use arrows to move.',
                  description='Collect stars.', remaining_s=600)
    policy = Agent()
    turns = []
    for stamp in (0., 1., 18., 23.):
        obs = Observation(image.getvalue(), stamp, hidden)
        start = time.perf_counter()
        act = policy.step(obs, game_tools(480, 320))
        validate(act.tool, act.params, 480, 320)
        turns.append(dict(mode=policy.general.mode, action=act.tool, params=act.params, images=MANAGER.last_images,
                          response=MANAGER.last_reply, seconds=round(time.perf_counter()-start, 3)))
    modes = [turn['mode'].split('+')[0] for turn in turns]
    assert modes[:2] == ['plan', 'act'] and modes[-1] == 'plan' and turns[-1]['images'] == 3, turns
    assert all(any(t['mode'].split('+')[0] == mode and t['response'] for t in turns)
               for mode in ('plan', 'act')), turns
    assert any(t['mode'] in ('plan', 'act') and t['response'] for t in turns), turns
    first = turns[0]
    pointer_steps = first['params'].get('steps', [first['params']])
    first['menu_hit'] = any(100 <= step.get('x', -1) <= 270 and 140 <= step.get('y', -1) <= 210
                            for step in pointer_steps)
    print('Unknown-game planner and actor:', json.dumps(turns), flush=True)
    Path('/kaggle/working/diagnostic.json').write_text(json.dumps(turns))

In [ ]:
from kaggle_evaluation.gga.gga_inference_server import GGAInferenceServer
Path('/kaggle/working/agent.py').write_text('from __main__ import Agent\n',encoding='utf-8')
GGAInferenceServer().run()

## 16. Credits for this agent, and the licence
The agent code is the work of **lightningv08** and the authors it adapts (above), published under the **Apache License 2.0** (http://www.apache.org/licenses/LICENSE-2.0). We changed nothing in it; the notebook around it (sections 1 to 14) is ours.